# 1. Objective and Scope

## 1.1 Objective

The objective of this notebook is to transform the validated document page records produced by the existing ingestion pipeline into smaller, coherent, traceable text chunks suitable for subsequent embedding and retrieval.

The implementation will explore and compare two chunking strategies:

* **Fixed-Window Chunking:** Divides page content into consecutive character-based windows with configurable overlap.
* **Structure-Aware Recursive Chunking:** Splits content using meaningful document boundaries, prioritizing Markdown sections, paragraphs, lines, and sentences.

## 1.2 Existing Infrastructure

This project already provides a document ingestion layer responsible for:

* Document discovery and text extraction.
* Document and page schema validation.
* Document identification and content hashing.
* Duplicate and unchanged-document detection.
* JSONL persistence.

These capabilities will be reused without reimplementation.

The chunking process will consume the existing:

* `DocumentRecord` records from `documents.jsonl`.
* `PageRecord` records from `pages.jsonl`.

Raw source files will not be rediscovered or re-extracted.

## 1.3 Scope

This notebook covers:

1. Loading and auditing existing processed records.
2. Defining the chunk data contract and configuration.
3. Implementing fixed-window chunking.
4. Parsing Markdown document structure.
5. Implementing structure-aware recursive chunking.
6. Preserving source provenance and character offsets.
7. Generating deterministic chunk identifiers.
8. Comparing chunking strategies.
9. Persisting and validating chunk records.
10. Evaluating implementation correctness through automated tests.

## 1.4 Initial Configuration

The initial experimental configuration is:

| Parameter         |            Initial Value |
| ----------------- | -----------------------: |
| Target Chunk Size |         1,200 characters |
| Target Overlap    |           180 characters |
| Chunk Length Unit | Python string characters |
| Input Unit        |    Individual PageRecord |
| Output Format     |                    JSONL |

These values are provisional experimental settings. They are not assumed to be optimal and will require validation against the selected embedding model's tokenizer.

## 1.5 Scope Exclusions

The following capabilities are outside the scope of this notebook:

* Document discovery and extraction.
* PDF parsing.
* Embedding generation.
* Vector database construction.
* Similarity search and retrieval.
* Reranking and query rewriting.
* LLM-based question answering.

## 1.6 Expected Output

The primary output is:

`data/knowledge_base/processed/chunks.jsonl`

Every generated chunk must retain its document identity, page number, source character offsets, content hash, chunking strategy, and configuration metadata.

The final objective is to produce reproducible, validated, source-traceable chunks ready for the embedding stage.


# 2. Load Existing Processed Records

## 2.1 Objective

Load the previously ingested document and page records from the existing JSONL storage.

This step reuses the ingestion module's existing loading functions and Pydantic models. No document discovery, extraction, or ingestion operations are performed.

## 2.2 Input Files

* `data/knowledge_base/processed/documents.jsonl`
* `data/knowledge_base/processed/pages.jsonl`

## 2.3 Loading Strategy

The existing functions are responsible for reading JSONL records and validating them against their corresponding Pydantic models:

* `load_existing_documents()`
* `load_existing_pages()`

The resulting objects will be used directly by the chunking experiments.

## 2.4 Expected Data Structures

* Documents: A dictionary indexed by `document_id`.
* Pages: A list of `PageRecord` objects.

## 2.5 Validation Objective

Confirm that both datasets are successfully loaded and that the returned objects use the expected model types.


In [1]:
import sys
from pathlib import Path

# Locate the project root from the current working directory.
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = next(
    (
        path
        for path in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (path / "app" / "ingestion").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Expected an 'app/ingestion' directory."
    )

# Add the project root to Python's module search path.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import existing ingestion models and loaders.
from app.ingestion.models import DocumentRecord, PageRecord
from app.ingestion.pipeline import (
    load_existing_documents,
    load_existing_pages,
)

# Define processed data paths.
PROCESSED_DIR = PROJECT_ROOT / "data" / "knowledge_base" / "processed"

DOCUMENTS_PATH = PROCESSED_DIR / "documents.jsonl"
PAGES_PATH = PROCESSED_DIR / "pages.jsonl"

# Load existing records.
documents = load_existing_documents(DOCUMENTS_PATH)
pages = load_existing_pages(PAGES_PATH)

# Display loading results.
print("PROJECT CONFIGURATION")
print("=" * 50)
print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {CURRENT_DIR}")

print("\nDOCUMENT LOADING")
print("=" * 50)
print(f"Documents loaded: {len(documents)}")
print(f"Pages loaded: {len(pages)}")

print("\nMODEL VALIDATION")
print("=" * 50)
print(
    "All documents are DocumentRecord:",
    all(isinstance(record, DocumentRecord) for record in documents.values()),
)
print(
    "All pages are PageRecord:",
    all(isinstance(record, PageRecord) for record in pages),
)

print("\nINPUT FILES")
print("=" * 50)
print(f"Documents: {DOCUMENTS_PATH}")
print(f"Pages: {PAGES_PATH}")

PROJECT CONFIGURATION
Project root: d:\Projects\Local Mini RAG Document QA System
Working directory: d:\Projects\Local Mini RAG Document QA System\notebooks

DOCUMENT LOADING
Documents loaded: 6
Pages loaded: 6

MODEL VALIDATION
All documents are DocumentRecord: True
All pages are PageRecord: True

INPUT FILES
Documents: d:\Projects\Local Mini RAG Document QA System\data\knowledge_base\processed\documents.jsonl
Pages: d:\Projects\Local Mini RAG Document QA System\data\knowledge_base\processed\pages.jsonl


### Result Interpretation — Loading Existing Records

The processed ingestion records were loaded successfully.

| Metric                    | Result |
| ------------------------- | -----: |
| Documents loaded          |      6 |
| Pages loaded              |      6 |
| DocumentRecord validation | Passed |
| PageRecord validation     | Passed |

The notebook successfully located the project root and imported the existing ingestion modules.

**Finding:** The chunking stage can reuse the existing processed records directly. Repeating document discovery or text extraction is unnecessary.

**Next Step:** Validate the integrity, relationships, and text characteristics of the loaded corpus before implementing chunking strategies.


# 3. Input Validation and Corpus Audit

## Objective

Validate the integrity and consistency of the previously processed document and page records.

This audit examines:

1. Document identifiers and content integrity.
2. Page identifiers and document references.
3. Empty or invalid text records.
4. Page numbering and document relationships.
5. Corpus-level text statistics.

## Validation Principles

* Every document must have a unique identifier.
* Every page must reference an existing document.
* Page content must be a valid string.
* Empty pages must be identified.
* Page numbering must be inspected before defining chunk boundaries.
* All statistics must be calculated from the actual processed corpus.

No document extraction or text transformation will be performed during this stage.

In [2]:
from collections import Counter
from statistics import mean

# --------------------------------------------------
# 1. Document Integrity
# --------------------------------------------------

document_ids = [document.document_id for document in documents.values()]

document_id_counts = Counter(document_ids)

duplicate_document_ids = [
    document_id for document_id, count in document_id_counts.items() if count > 1
]

empty_document_content = [
    document.document_id
    for document in documents.values()
    if not document.content.strip()
]

# --------------------------------------------------
# 2. Page Integrity
# --------------------------------------------------

page_document_ids = [page.document_id for page in pages]

page_id_counts = Counter((page.document_id, page.page_number) for page in pages)

duplicate_pages = [page_id for page_id, count in page_id_counts.items() if count > 1]

missing_document_references = [
    page.document_id for page in pages if page.document_id not in documents
]

empty_pages = [
    (page.document_id, page.page_number) for page in pages if not page.content.strip()
]

# --------------------------------------------------
# 3. Page Numbering
# --------------------------------------------------

pages_by_document = {}

for page in pages:
    pages_by_document.setdefault(page.document_id, []).append(page.page_number)

page_numbering = {}

for document_id, page_numbers in pages_by_document.items():
    sorted_numbers = sorted(page_numbers)

    page_numbering[document_id] = {
        "page_numbers": sorted_numbers,
        "is_sorted_contiguous": (
            sorted_numbers
            == list(
                range(
                    sorted_numbers[0],
                    sorted_numbers[0] + len(sorted_numbers),
                )
            )
        ),
    }

# --------------------------------------------------
# 4. Text Statistics
# --------------------------------------------------

page_lengths = [len(page.content) for page in pages]

document_lengths = [len(document.content) for document in documents.values()]

total_page_characters = sum(page_lengths)
total_document_characters = sum(document_lengths)

# --------------------------------------------------
# 5. Document-to-Page Relationships
# --------------------------------------------------

pages_per_document = Counter(page.document_id for page in pages)

documents_without_pages = [
    document_id for document_id in documents if pages_per_document[document_id] == 0
]

# --------------------------------------------------
# 6. Audit Report
# --------------------------------------------------

print("DOCUMENT INTEGRITY")
print("=" * 55)
print(f"Unique document IDs: {len(set(document_ids))}")
print(f"Duplicate document IDs: {duplicate_document_ids}")
print(f"Documents with empty content: {empty_document_content}")

print("\nPAGE INTEGRITY")
print("=" * 55)
print(f"Total pages: {len(pages)}")
print(f"Duplicate pages: {duplicate_pages}")
print(f"Missing document references: {missing_document_references}")
print(f"Empty pages: {empty_pages}")

print("\nPAGE NUMBERING")
print("=" * 55)

for document_id, result in page_numbering.items():
    print(f"{document_id}: {result}")

print("\nDOCUMENT-PAGE RELATIONSHIPS")
print("=" * 55)
print(f"Documents without pages: {documents_without_pages}")

for document_id, count in pages_per_document.items():
    print(f"{document_id}: {count} page(s)")

print("\nCORPUS STATISTICS")
print("=" * 55)
print(f"Total document characters: {total_document_characters:,}")
print(f"Total page characters: {total_page_characters:,}")
print(f"Minimum page length: {min(page_lengths):,}")
print(f"Maximum page length: {max(page_lengths):,}")
print(f"Average page length: {mean(page_lengths):,.2f}")
print(f"Minimum document length: {min(document_lengths):,}")
print(f"Maximum document length: {max(document_lengths):,}")
print(f"Average document length: {mean(document_lengths):,.2f}")

DOCUMENT INTEGRITY
Unique document IDs: 6
Duplicate document IDs: []
Documents with empty content: []

PAGE INTEGRITY
Total pages: 6
Duplicate pages: []
Missing document references: []
Empty pages: []

PAGE NUMBERING
5eaa2f3a4137bbd8: {'page_numbers': [1], 'is_sorted_contiguous': True}
4bc29707830b7f97: {'page_numbers': [1], 'is_sorted_contiguous': True}
6e0ae6c5f11c01b6: {'page_numbers': [1], 'is_sorted_contiguous': True}
1f8658d7b4d1d931: {'page_numbers': [1], 'is_sorted_contiguous': True}
17f4f629bc6c06aa: {'page_numbers': [1], 'is_sorted_contiguous': True}
b8c040cc92d5243c: {'page_numbers': [1], 'is_sorted_contiguous': True}

DOCUMENT-PAGE RELATIONSHIPS
Documents without pages: []
5eaa2f3a4137bbd8: 1 page(s)
4bc29707830b7f97: 1 page(s)
6e0ae6c5f11c01b6: 1 page(s)
1f8658d7b4d1d931: 1 page(s)
17f4f629bc6c06aa: 1 page(s)
b8c040cc92d5243c: 1 page(s)

CORPUS STATISTICS
Total document characters: 118,204
Total page characters: 118,204
Minimum page length: 9,497
Maximum page length: 32,36

### Result Interpretation — Input Validation and Corpus Audit

The corpus passed all integrity checks.

| Metric                      |    Result |
| --------------------------- | --------: |
| Documents                   |         6 |
| Pages                       |         6 |
| Total characters            |   118,204 |
| Minimum page length         |     9,497 |
| Maximum page length         |    32,360 |
| Average page length         | 19,700.67 |
| Duplicate document IDs      |         0 |
| Duplicate pages             |         0 |
| Missing document references |         0 |
| Empty documents or pages    |         0 |

### Structural Findings

* Each document currently contains exactly one logical page.
* All page numbers start at 1.
* Every page references an existing document.
* Document content and page content have matching total character counts.

### Technical Findings

1. The corpus is structurally valid and ready for chunking experiments.
2. Chunk provenance must preserve document IDs and page numbers.
3. Document length varies significantly, so chunking behavior must be evaluated across different input sizes.
4. Multi-page document handling should remain supported by the implementation, even though the current corpus does not exercise that case.

**Conclusion:** No ingestion or data-integrity issues were detected. The existing processed records can be used as the input for the chunking pipeline.

**Next Step:** Inspect Markdown structure to understand headings, paragraphs, lists, tables, and code blocks before selecting chunk boundaries.


# 4. Understanding Document Structure

## Objective

Inspect the actual Markdown structure of the processed corpus before implementing chunking algorithms.

## Structural Elements

The audit focuses on:

1. ATX headings (`#`, `##`, `###`, etc.).
2. Paragraphs.
3. Unordered and ordered lists.
4. Markdown tables.
5. Fenced code blocks using backticks or tildes.
6. Horizontal rules.

## Why Structure Matters

Chunk boundaries should preserve meaningful relationships between document elements.

For example, a heading should remain associated with its corresponding content, list items should preferably remain intact, and code blocks should not be interpreted as ordinary Markdown headings.

## Scope

This section only inspects and reports document structure. It does not modify the source text or implement chunking.


In [ ]:
import re
from collections import Counter

# Markdown structure patterns.
PATTERNS = {
    "headings": re.compile(r"^\s{0,3}#{1,6}\s+\S", re.MULTILINE),
    "unordered_lists": re.compile(r"^\s*[-+*]\s+\S", re.MULTILINE),
    "ordered_lists": re.compile(r"^\s*\d+[.)]\s+\S", re.MULTILINE),
    "table_separator": re.compile(
        r"^\s*\|?\s*:?-{3,}:?\s*(\|\s*:?-{3,}:?\s*)+\|?\s*$",
        re.MULTILINE,
    ),
    "horizontal_rules": re.compile(
        r"^\s{0,3}((\*\s*){3,}|(-\s*){3,}|(_\s*){3,})$",
        re.MULTILINE,
    ),
}


def inspect_markdown_structure(content: str) -> dict:
    lines = content.splitlines()

    fenced_code_blocks = 0
    inside_code_block = False
    fence_marker = None

    code_lines = 0
    structural_counts = Counter()

    for line in lines:
        stripped = line.lstrip()

        # Detect fenced code block boundaries.
        fence_match = re.match(r"^(`{3,}|~{3,})", stripped)

        if fence_match:
            marker = fence_match.group(1)

            if not inside_code_block:
                inside_code_block = True
                fence_marker = marker[0]
                fenced_code_blocks += 1
                continue

            if marker[0] == fence_marker:
                inside_code_block = False
                fence_marker = None
                continue

        if inside_code_block:
            code_lines += 1
            continue

        if not line.strip():
            continue

        if PATTERNS["headings"].match(line):
            structural_counts["headings"] += 1

        if PATTERNS["unordered_lists"].match(line):
            structural_counts["unordered_lists"] += 1

        if PATTERNS["ordered_lists"].match(line):
            structural_counts["ordered_lists"] += 1

        if "|" in line and line.count("|") >= 2:
            structural_counts["table_like_lines"] += 1

        if PATTERNS["table_separator"].match(line):
            structural_counts["table_separator_lines"] += 1

        if PATTERNS["horizontal_rules"].match(line):
            structural_counts["horizontal_rules"] += 1

    return {
        "total_lines": len(lines),
        "non_empty_lines": sum(bool(line.strip()) for line in lines),
        "fenced_code_blocks": fenced_code_blocks,
        "code_lines": code_lines,
        "unclosed_code_fence": inside_code_block,
        **dict(structural_counts),
    }


# Inspect all actual pages.
structure_reports = []

for page in pages:
    report = inspect_markdown_structure(page.content)

    structure_reports.append(
        {
            "document_id": page.document_id,
            "page_number": page.page_number,
            **report,
        }
    )


# Display per-document results.
print("MARKDOWN STRUCTURE AUDIT")
print("=" * 70)

for report in structure_reports:
    print(f"\nDocument: {report['document_id']}")
    print(f"Page: {report['page_number']}")

    for key, value in report.items():
        if key not in {"document_id", "page_number"}:
            print(f"{key}: {value}")


# Aggregate structural statistics.
aggregate = Counter()

for report in structure_reports:
    for key, value in report.items():
        if key not in {"document_id", "page_number"}:
            if isinstance(value, int):
                aggregate[key] += value

print("\nCORPUS STRUCTURE SUMMARY")
print("=" * 70)

for key, value in aggregate.items():
    print(f"{key}: {value}")

MARKDOWN STRUCTURE AUDIT

Document: 5eaa2f3a4137bbd8
Page: 1
total_lines: 374
non_empty_lines: 257
fenced_code_blocks: 6
code_lines: 42
unclosed_code_fence: False
headings: 30
unordered_lists: 52
table_like_lines: 51
table_separator_lines: 5
ordered_lists: 3

Document: 4bc29707830b7f97
Page: 1
total_lines: 259
non_empty_lines: 186
fenced_code_blocks: 2
code_lines: 2
unclosed_code_fence: False
headings: 22
unordered_lists: 28
horizontal_rules: 2
table_like_lines: 21
table_separator_lines: 3
ordered_lists: 13

Document: 6e0ae6c5f11c01b6
Page: 1
total_lines: 623
non_empty_lines: 431
fenced_code_blocks: 10
code_lines: 56
unclosed_code_fence: False
headings: 65
horizontal_rules: 1
unordered_lists: 50
ordered_lists: 49
table_like_lines: 100
table_separator_lines: 9

Document: 1f8658d7b4d1d931
Page: 1
total_lines: 565
non_empty_lines: 344
fenced_code_blocks: 0
code_lines: 0
unclosed_code_fence: False
headings: 45
unordered_lists: 75
horizontal_rules: 15
table_like_lines: 63
table_separator_li

### Result Interpretation — Markdown Structure Audit

The Markdown structure audit identified a highly structured corpus.

| Structural Element    | Count |
| --------------------- | ----: |
| Headings              |   205 |
| Unordered list items  |   314 |
| Ordered list items    |    86 |
| Table-like lines      |   340 |
| Table separator lines |    31 |
| Fenced code blocks    |    34 |
| Code lines            |   159 |
| Horizontal rules      |    18 |
| Total lines           | 2,368 |
| Non-empty lines       | 1,635 |

### Technical Findings

1. The corpus contains a substantial heading hierarchy that can support section-aware chunk metadata.
2. Lists represent a significant portion of the content and should be preserved as logical units whenever possible.
3. Table detection requires more precise parsing than counting pipe characters.
4. Code fences must be tracked to prevent code content from being misinterpreted as Markdown structure.
5. All inspected documents have balanced fenced code blocks according to the initial structural audit.

### Implementation Implications

The chunking implementation should:

* Maintain heading hierarchy and section paths.
* Preserve paragraph boundaries.
* Respect ordered and unordered list boundaries.
* Recognize Markdown tables using structural validation.
* Support backtick and tilde code fences.
* Split oversized structural elements without losing text or provenance.

**Conclusion:** The corpus requires structure-aware chunking in addition to a fixed-window baseline. The structural audit provides the evidence needed to design the Markdown parser.

**Next Step:** Define chunking fundamentals, configuration parameters, and measurable constraints before implementing the first strategy.


# 5. Chunking Fundamentals and Configuration

## Objective

Define the initial chunking configuration and evaluation criteria before implementing any chunking strategy.

## Chunking Parameters

| Parameter            | Initial Value | Description                               |
| -------------------- | ------------: | ----------------------------------------- |
| CHUNK_SIZE_CHARS     |          1200 | Target chunk length in Unicode characters |
| CHUNK_OVERLAP_CHARS  |           180 | Target overlap between consecutive chunks |
| MIN_CHUNK_SIZE_CHARS |           100 | Initial minimum size for ordinary chunks  |
| CHUNKING_VERSION     |             1 | Chunking algorithm version                |

These values are experimental starting points, not proven optimal settings.

## Measurement Criteria

The experiments will evaluate:

1. Total number of generated chunks.
2. Average, minimum, and maximum chunk lengths.
3. Actual overlap between consecutive chunks.
4. Oversized chunk frequency.
5. Source text coverage and integrity.
6. Preservation of structural boundaries.
7. Deterministic output across repeated runs.

## Important Constraints

* Character counts use Python Unicode string indexing.
* Character counts are not equivalent to tokenizer token counts.
* Chunk content must preserve its exact source substring.
* Chunk offsets must refer to the original page content.
* Structure-aware splitting may produce chunks that exceed the target size when preserving a logical unit is preferable.
* No embedding or retrieval evaluation is performed in this stage.

## Experimental Principle

The fixed-window strategy provides a baseline. The structure-aware recursive strategy will be compared against it using the same corpus and measurable integrity criteria.


In [ ]:
# Initial chunking configuration.

CHUNK_SIZE_CHARS = 1200
CHUNK_OVERLAP_CHARS = 180
MIN_CHUNK_SIZE_CHARS = 100
CHUNKING_VERSION = 1

# Derived configuration.
EFFECTIVE_STEP_CHARS = CHUNK_SIZE_CHARS - CHUNK_OVERLAP_CHARS

# Configuration validation.
assert CHUNK_SIZE_CHARS > 0
assert 0 <= CHUNK_OVERLAP_CHARS < CHUNK_SIZE_CHARS
assert 0 < MIN_CHUNK_SIZE_CHARS <= CHUNK_SIZE_CHARS

# Display configuration.
print("CHUNKING CONFIGURATION")
print("=" * 55)
print(f"Target chunk size: {CHUNK_SIZE_CHARS:,} characters")
print(f"Target overlap: {CHUNK_OVERLAP_CHARS:,} characters")
print(f"Effective step: {EFFECTIVE_STEP_CHARS:,} characters")
print(f"Minimum chunk size: {MIN_CHUNK_SIZE_CHARS:,} characters")
print(f"Chunking version: {CHUNKING_VERSION}")

print("\nCONFIGURATION VALIDATION")
print("=" * 55)
print("Chunk size is positive: True")
print("Overlap is smaller than chunk size: True")
print("Minimum size is valid: True")

CHUNKING CONFIGURATION
Target chunk size: 1,200 characters
Target overlap: 180 characters
Effective step: 1,020 characters
Minimum chunk size: 100 characters
Chunking version: 1

CONFIGURATION VALIDATION
Chunk size is positive: True
Overlap is smaller than chunk size: True
Minimum size is valid: True


### Result Interpretation — Chunking Configuration

The initial chunking configuration was defined and validated successfully.

| Parameter          |            Value |
| ------------------ | ---------------: |
| Target chunk size  | 1,200 characters |
| Target overlap     |   180 characters |
| Effective step     | 1,020 characters |
| Minimum chunk size |   100 characters |
| Chunking version   |                1 |

All configuration assertions passed.

### Technical Findings

1. The configuration uses character-based measurements.
2. The fixed-window strategy will use a 1,020-character step.
3. The overlap value is a target for the fixed-window baseline.
4. Structure-aware chunking may produce different actual chunk sizes and overlaps to preserve logical boundaries.
5. These values are experimental starting points and require evaluation.

**Conclusion:** The initial configuration is valid and ready to be used by the chunking implementation.

**Next Step:** Define a reusable ChunkRecord schema that preserves source content, offsets, structural metadata, and provenance.


# 6. Chunk Record Schema

## Objective

Design a reusable and validated data model for representing individual text chunks.

The `ChunkRecord` model will preserve the relationship between each generated chunk and its original source document and page.

## Schema Definition

| Field              | Type             | Description                                                 |
| ------------------ | ---------------- | ----------------------------------------------------------- |
| `chunk_id`         | `str`            | Unique deterministic identifier                             |
| `document_id`      | `str`            | Original source document identifier                         |
| `page_number`      | `int`            | Original source page number                                 |
| `chunk_index`      | `int`            | Sequential chunk position within the page, starting at zero |
| `content`          | `str`            | Exact text extracted from the source page                   |
| `char_start`       | `int`            | Inclusive starting character offset                         |
| `char_end`         | `int`            | Exclusive ending character offset                           |
| `content_hash`     | `str`            | SHA-256 hash of the chunk content                           |
| `section_path`     | `list[str]`      | Markdown heading hierarchy associated with the chunk        |
| `strategy`         | `str`            | Chunking strategy used to generate the chunk                |
| `config_signature` | `str`            | Identifier representing the chunking configuration          |
| `chunking_version` | `int`            | Version of the chunking algorithm                           |
| `metadata`         | `dict[str, Any]` | Additional chunk information                                |

## Validation Rules

The model must enforce the following constraints:

1. Identifiers and strategy names must not be empty.
2. Page numbers must be greater than or equal to 1.
3. Chunk indices must start from zero or greater.
4. Character offsets must be non-negative.
5. The ending offset must be greater than the starting offset.
6. The content hash must contain exactly 64 hexadecimal characters.
7. The chunk content length must equal the difference between its ending and starting offsets.
8. Unknown fields must be rejected.
9. Mutable default values must use `default_factory`.

## Source Offset Convention

Character offsets follow Python string slicing conventions.

* `char_start` is inclusive.
* `char_end` is exclusive.

The source integrity invariant is:

`chunk.content == page.content[chunk.char_start:chunk.char_end]`

The model can validate offset length consistency. Actual source substring equality will be verified separately against the original `PageRecord`.

## Design Principles

* Follow the existing Pydantic model conventions used in `app/ingestion/models.py`.
* Keep chunking models independent from ingestion models.
* Preserve source provenance.
* Support deterministic chunk identification.
* Keep the schema extensible through metadata without allowing arbitrary top-level fields.

## Expected Outcome

A reusable `ChunkRecord` model that provides a consistent data contract for all chunking strategies, validation tests, persistence operations, and future retrieval components.


In [ ]:
from app.chunking.models import ChunkRecord

print("CHUNK MODEL IMPORT")
print("=" * 50)
print("ChunkRecord imported successfully.")

print("\nMODEL FIELDS")
print("=" * 50)

for field_name, field_info in ChunkRecord.model_fields.items():
    print(f"{field_name}: {field_info.annotation}")

print("\nMODEL CONFIGURATION")
print("=" * 50)
print(f"Extra fields: {ChunkRecord.model_config['extra']}")
print(f"Whitespace stripping: {ChunkRecord.model_config['str_strip_whitespace']}")

CHUNK MODEL IMPORT
ChunkRecord imported successfully.

MODEL FIELDS
chunk_id: <class 'str'>
document_id: <class 'str'>
page_number: <class 'int'>
chunk_index: <class 'int'>
content: <class 'str'>
char_start: <class 'int'>
char_end: <class 'int'>
content_hash: <class 'str'>
section_path: list[str]
strategy: <class 'str'>
config_signature: <class 'str'>
chunking_version: <class 'int'>
metadata: dict[str, typing.Any]

MODEL CONFIGURATION
Extra fields: forbid
Whitespace stripping: False


### Result Interpretation — Chunk Record Schema

The `ChunkRecord` Pydantic model was created and imported successfully.

The model exposes 13 fields covering:

* Unique chunk identification.
* Source document and page provenance.
* Chunk ordering.
* Exact text content and character offsets.
* SHA-256 content hashing.
* Markdown section hierarchy.
* Chunking strategy and configuration metadata.

### Validation Configuration

| Setting              | Result     |
| -------------------- | ---------- |
| Extra fields         | Forbidden  |
| Whitespace stripping | Disabled   |
| Model import         | Successful |
| Field definitions    | Validated  |

### Technical Findings

1. The chunk schema follows the existing ingestion model conventions.
2. Source provenance is represented explicitly.
3. Character offsets use inclusive start and exclusive end semantics.
4. The model validates offset-length consistency.
5. Actual source substring equality will require a separate integrity test.

**Conclusion:** The initial `ChunkRecord` data contract is implemented and ready for unit testing.

### ChunkRecord Validation Tests

The `ChunkRecord` model passed all nine unit tests.

| Metric          |       Result |
| --------------- | -----------: |
| Tests collected |            9 |
| Tests passed    |            9 |
| Tests failed    |            0 |
| Execution time  | 0.30 seconds |

### Validation Findings

* Valid chunk records are accepted.
* Empty identifiers are rejected.
* Invalid page numbers and negative chunk indices are rejected.
* Invalid SHA-256 hashes are rejected.
* Inconsistent and reversed character offsets are rejected.
* Unknown fields are rejected.
* Mutable default values are independent across model instances.

**Conclusion:** The initial `ChunkRecord` model satisfies its tested validation requirements.

**Next Step:** Implement and evaluate the Fixed-Window Chunking baseline using the existing processed corpus.

# 7. Fixed-Window Chunking

## Objective

Implement a baseline chunking strategy that divides page content into fixed-length character windows with a configurable overlap.

## Algorithm Parameters

The implementation uses the initial experimental configuration:

* Target chunk size: 1,200 characters.
* Target overlap: 180 characters.
* Effective step: 1,020 characters.

## Algorithm Behavior

1. Start at character offset zero.
2. Extract a window up to the configured chunk size.
3. Record the inclusive starting offset and exclusive ending offset.
4. Advance by the effective step.
5. Continue until the entire page content has been covered.
6. Preserve the original text without stripping or modifying whitespace.

## Source Integrity

Every generated chunk must satisfy:

`chunk.content == page.content[chunk.char_start:chunk.char_end]`

The algorithm must not introduce gaps or modify the original page content.

## Scope

This baseline strategy does not interpret Markdown headings, paragraphs, lists, tables, or code blocks.

Structural preservation will be addressed by the recursive chunking strategy.

## Expected Measurements

* Number of generated chunks.
* Average, minimum, and maximum chunk length.
* Actual overlap between consecutive chunks.
* Source coverage.
* Offset integrity.


### Fixed-Window Algorithm — Unit Experiment

## Objective

Verify the correctness of the fixed-window splitting algorithm using a controlled synthetic text input.

## Test Configuration

| Parameter      |            Value |
| -------------- | ---------------: |
| Input length   | 2,500 characters |
| Chunk size     | 1,000 characters |
| Overlap        |   200 characters |
| Effective step |   800 characters |

## Validation Criteria

1. Every chunk must contain the exact corresponding source substring.
2. The first chunk must start at offset zero.
3. The final chunk must end at the source text length.
4. Consecutive chunks must have no uncovered gaps.
5. The actual overlap must match the configured overlap when consecutive chunks are full-sized.
6. Chunk offsets must be valid and ordered.

## Experimental Input

A synthetic text string is used to isolate algorithm behavior from document-specific Markdown structure.

No real corpus data is modified during this experiment.

In [ ]:
from app.chunking.strategies import fixed_window_chunk

# Experiment configuration
sample_text = "A" * 2500
chunk_size = 1000
overlap = 200

# Generate chunks
chunks = fixed_window_chunk(
    content=sample_text,
    chunk_size=chunk_size,
    overlap=overlap,
)

# Display experiment configuration
print("FIXED-WINDOW EXPERIMENT")
print("=" * 55)

print(f"Input length: {len(sample_text)}")
print(f"Chunk size: {chunk_size}")
print(f"Overlap: {overlap}")
print(f"Effective step: {chunk_size - overlap}")
print(f"Generated chunks: {len(chunks)}")

# Display chunk details
print("\nCHUNK DETAILS")
print("=" * 55)

for index, (start, end, chunk_content) in enumerate(chunks):
    print(f"Chunk {index}: start={start}, end={end}, length={len(chunk_content)}")

# Validate chunk integrity
print("\nINTEGRITY VALIDATION")
print("=" * 55)

substring_integrity = all(
    chunk_content == sample_text[start:end] for start, end, chunk_content in chunks
)

first_chunk_starts_at_zero = chunks[0][0] == 0 if chunks else False

last_chunk_ends_at_source_length = (
    chunks[-1][1] == len(sample_text) if chunks else False
)

no_gaps_between_chunks = all(
    current_start <= previous_end
    for (_, previous_end, _), (current_start, _, _) in zip(chunks, chunks[1:])
)

complete_source_coverage = (
    first_chunk_starts_at_zero
    and last_chunk_ends_at_source_length
    and no_gaps_between_chunks
)

# Calculate actual overlaps
actual_overlaps = [
    previous_end - current_start
    for (_, previous_end, _), (current_start, _, _) in zip(chunks, chunks[1:])
]

full_sized_overlaps_are_correct = all(
    actual_overlap == overlap
    for actual_overlap in actual_overlaps
    if actual_overlap > 0
)

# Display validation results
print(f"All chunks match source substrings: {substring_integrity}")
print(f"First chunk starts at zero: {first_chunk_starts_at_zero}")
print(f"Last chunk ends at source length: {last_chunk_ends_at_source_length}")
print(f"No gaps between chunks: {no_gaps_between_chunks}")
print(f"Complete source coverage: {complete_source_coverage}")
print(f"Actual overlaps: {actual_overlaps}")
print(
    "All full-sized consecutive overlaps are correct: "
    f"{full_sized_overlaps_are_correct}"
)

# Final assertion checks
assert substring_integrity
assert complete_source_coverage
assert full_sized_overlaps_are_correct

print("\nExperiment completed successfully.")

FIXED-WINDOW EXPERIMENT
Input length: 2500
Chunk size: 1000
Overlap: 200
Effective step: 800
Generated chunks: 3

CHUNK DETAILS
Chunk 0: start=0, end=1000, length=1000
Chunk 1: start=800, end=1800, length=1000
Chunk 2: start=1600, end=2500, length=900

INTEGRITY VALIDATION
All chunks match source substrings: True
First chunk starts at zero: True
Last chunk ends at source length: True
No gaps between chunks: True
Complete source coverage: True
Actual overlaps: [200, 200]
All full-sized consecutive overlaps are correct: True

Experiment completed successfully.


### Results and Interpretation

The fixed-window chunking experiment successfully divided a 2,500-character input into three overlapping chunks.

### Configuration

* Chunk size: 1,000 characters
* Overlap: 200 characters
* Effective step: 800 characters

### Observations

1. The algorithm generated exactly three chunks.
2. The first two chunks contain 1,000 characters each.
3. The final chunk contains 900 characters because it reaches the end of the source text.
4. The actual overlap between consecutive chunks is 200 characters.
5. Every chunk matches its corresponding source substring.
6. The generated chunks cover the entire source without gaps.

### Conclusion

The fixed-window implementation passed all integrity and coverage checks for this experiment.

The implementation is ready for edge-case testing. These results validate the tested configuration and input, not every possible input.


## Fixed-Window Chunking — Edge Cases

### Objective

Evaluate the behavior of the fixed-window chunking algorithm under different input conditions and configuration boundaries.

### Test Cases

The following cases will be evaluated:

1. **Empty Input:** Verify that an empty string produces no chunks.
2. **Short Input:** Verify that input shorter than the configured chunk size produces one chunk.
3. **Exact-Size Input:** Verify that input equal to the chunk size produces exactly one chunk.
4. **Slightly Larger Input:** Verify that input slightly larger than the chunk size is divided without losing characters.
5. **Zero Overlap:** Verify that consecutive chunks do not overlap.
6. **Maximum Valid Overlap:** Verify that an overlap of `chunk_size - 1` is accepted and handled correctly.
7. **Invalid Chunk Size:** Verify that zero or negative chunk sizes raise a `ValueError`.
8. **Invalid Overlap:** Verify that negative overlap or overlap greater than or equal to the chunk size raises a `ValueError`.

### Validation Criteria

* Empty input must return an empty list.
* Valid inputs must preserve complete source coverage.
* Every chunk must match its corresponding source substring.
* Invalid configurations must raise the expected exceptions.
* No redundant chunks should be generated after reaching the end of the source text.

### Expected Outcome

The implementation should behave consistently across normal, boundary, and invalid input conditions.

Actual results and conclusions will be documented after executing the test cases.


In [ ]:
from app.chunking.strategies import fixed_window_chunk


def run_edge_case(name, content, chunk_size, overlap):
    chunks = fixed_window_chunk(
        content=content,
        chunk_size=chunk_size,
        overlap=overlap,
    )

    print(f"\n{name}")
    print("-" * 50)
    print(f"Input length: {len(content)}")
    print(f"Generated chunks: {len(chunks)}")

    for index, (start, end, chunk_content) in enumerate(chunks):
        print(f"Chunk {index}: start={start}, end={end}, length={len(chunk_content)}")

    return chunks


# Case 1: Empty input
empty_chunks = run_edge_case(
    "CASE 1: Empty Input",
    "",
    1000,
    200,
)

# Case 2: Input shorter than chunk size
short_chunks = run_edge_case(
    "CASE 2: Input Shorter Than Chunk Size",
    "A" * 500,
    1000,
    200,
)

# Case 3: Input exactly equal to chunk size
exact_chunks = run_edge_case(
    "CASE 3: Input Equal To Chunk Size",
    "B" * 1000,
    1000,
    200,
)

# Case 4: Input slightly larger than chunk size
larger_chunks = run_edge_case(
    "CASE 4: Input Slightly Larger Than Chunk Size",
    "C" * 1001,
    1000,
    200,
)

# Case 5: Zero overlap
zero_overlap_chunks = run_edge_case(
    "CASE 5: Zero Overlap",
    "D" * 2500,
    1000,
    0,
)

# Case 6: Maximum valid overlap
maximum_overlap_chunks = run_edge_case(
    "CASE 6: Maximum Valid Overlap",
    "E" * 1500,
    1000,
    999,
)

# Validate expected behavior
assert empty_chunks == []
assert len(short_chunks) == 1
assert len(exact_chunks) == 1
assert len(larger_chunks) == 2
assert len(zero_overlap_chunks) == 3
assert len(maximum_overlap_chunks) == 501

print("\n" + "=" * 50)
print("EDGE-CASE ASSERTIONS PASSED")


CASE 1: Empty Input
--------------------------------------------------
Input length: 0
Generated chunks: 0

CASE 2: Input Shorter Than Chunk Size
--------------------------------------------------
Input length: 500
Generated chunks: 1
Chunk 0: start=0, end=500, length=500

CASE 3: Input Equal To Chunk Size
--------------------------------------------------
Input length: 1000
Generated chunks: 1
Chunk 0: start=0, end=1000, length=1000

CASE 4: Input Slightly Larger Than Chunk Size
--------------------------------------------------
Input length: 1001
Generated chunks: 2
Chunk 0: start=0, end=1000, length=1000
Chunk 1: start=800, end=1001, length=201

CASE 5: Zero Overlap
--------------------------------------------------
Input length: 2500
Generated chunks: 3
Chunk 0: start=0, end=1000, length=1000
Chunk 1: start=1000, end=2000, length=1000
Chunk 2: start=2000, end=2500, length=500

CASE 6: Maximum Valid Overlap
--------------------------------------------------
Input length: 1500
Gener

## Fixed-Window Edge Cases — Results and Interpretation

### Test Results

Six edge cases were executed successfully:

| Test Case             | Input Length | Generated Chunks | Result |
| --------------------- | -----------: | ---------------: | ------ |
| Empty Input           |            0 |                0 | Passed |
| Short Input           |          500 |                1 | Passed |
| Exact-Size Input      |        1,000 |                1 | Passed |
| Slightly Larger Input |        1,001 |                2 | Passed |
| Zero Overlap          |        2,500 |                3 | Passed |
| Maximum Valid Overlap |        1,500 |              501 | Passed |

### Key Observations

1. Empty input correctly produces an empty list.
2. Inputs shorter than or equal to the chunk size produce exactly one chunk.
3. Inputs slightly larger than the chunk size are divided while preserving the remaining characters.
4. Zero overlap produces consecutive non-overlapping chunks.
5. Maximum valid overlap produces 501 chunks for a 1,500-character input.

### Important Finding: Maximum Overlap

With a chunk size of 1,000 and an overlap of 999, the effective step becomes 1 character.

Although this configuration is mathematically valid, it generates excessive redundancy and is inefficient for practical document processing.

### Conclusion

All six tested edge cases passed their assertions.

The results confirm the expected behavior for the tested inputs. However, invalid configuration handling still requires explicit testing before considering edge-case validation complete.


## Invalid Configuration Tests

### Objective

Validate that the fixed-window chunking function rejects invalid parameter configurations before processing the input text.

### Test Cases

The following invalid configurations will be tested:

| Test Case                  | Chunk Size | Overlap | Expected Behavior  |
| -------------------------- | ---------: | ------: | ------------------ |
| Zero chunk size            |          0 |       0 | Raise `ValueError` |
| Negative chunk size        |       -100 |       0 | Raise `ValueError` |
| Negative overlap           |        100 |      -1 | Raise `ValueError` |
| Overlap equals chunk size  |        100 |     100 | Raise `ValueError` |
| Overlap exceeds chunk size |        100 |     120 | Raise `ValueError` |

### Validation Criteria

All invalid configurations must raise a `ValueError` with an explanatory error message.

The function must not silently accept invalid parameters or proceed with chunk generation.

### Expected Outcome

The fixed-window chunking function should enforce the following constraints:

* `chunk_size > 0`
* `overlap >= 0`
* `overlap < chunk_size`

These constraints prevent invalid window configurations and ensure that the sliding-window step remains positive.


In [13]:
from app.chunking.strategies import fixed_window_chunk


invalid_cases = [
    ("Zero chunk size", "A" * 100, 0, 0),
    ("Negative chunk size", "A" * 100, -100, 0),
    ("Negative overlap", "A" * 100, 100, -1),
    ("Overlap equal to chunk size", "A" * 100, 100, 100),
    ("Overlap greater than chunk size", "A" * 100, 100, 120),
]

print("INVALID CONFIGURATION TESTS")
print("=" * 55)

for name, content, chunk_size, overlap in invalid_cases:
    try:
        fixed_window_chunk(
            content=content,
            chunk_size=chunk_size,
            overlap=overlap,
        )
    except ValueError as error:
        print(f"{name}: PASSED")
        print(f"  Expected exception: {type(error).__name__}")
        print(f"  Message: {error}")
    else:
        raise AssertionError(
            f"{name}: Expected ValueError, but no exception was raised."
        )

print("\nAll invalid configuration tests passed.")

INVALID CONFIGURATION TESTS
Zero chunk size: PASSED
  Expected exception: ValueError
  Message: chunk_size must be greater than zero.
Negative chunk size: PASSED
  Expected exception: ValueError
  Message: chunk_size must be greater than zero.
Negative overlap: PASSED
  Expected exception: ValueError
  Message: overlap must be non-negative and smaller than chunk_size.
Overlap equal to chunk size: PASSED
  Expected exception: ValueError
  Message: overlap must be non-negative and smaller than chunk_size.
Overlap greater than chunk size: PASSED
  Expected exception: ValueError
  Message: overlap must be non-negative and smaller than chunk_size.

All invalid configuration tests passed.


### Result Interpretation

All five invalid configuration tests passed successfully.

| Test Case                       | Actual Result       | Status |
| ------------------------------- | ------------------- | ------ |
| Zero chunk size                 | Raised `ValueError` | PASS   |
| Negative chunk size             | Raised `ValueError` | PASS   |
| Negative overlap                | Raised `ValueError` | PASS   |
| Overlap equal to chunk size     | Raised `ValueError` | PASS   |
| Overlap greater than chunk size | Raised `ValueError` | PASS   |

### Key Findings

1. The function correctly rejects zero and negative chunk sizes.
2. Negative overlap values are rejected.
3. Overlap values equal to or greater than the chunk size are rejected.
4. Each invalid configuration produces an explicit and meaningful error message.
5. No invalid configuration was silently accepted.

### Conclusion

The fixed-window chunking function successfully enforces all tested parameter constraints.

The validation rules guarantee that:

* `chunk_size > 0`
* `0 <= overlap < chunk_size`

Therefore, the sliding-window step (`chunk_size - overlap`) is always positive for valid configurations.

**Validation Status: PASSED**

All five invalid configuration tests passed. No implementation changes were required.


# 8. Markdown Structure Parsing

### Objective

Build a lightweight Markdown structure parser that identifies document elements and preserves their structural boundaries before applying structure-aware recursive chunking.

### Why Structure Parsing Matters

Fixed-window chunking splits text based on character positions without understanding its semantic or structural organization.

This can result in:

* Headings being separated from their associated content.
* Markdown tables being split across rows.
* Lists being divided in the middle of logical items.
* Code blocks being incorrectly fragmented.
* Heading-like text inside code blocks being misclassified as Markdown headings.

A structure-aware parser helps identify these elements before chunk generation.

### Elements to Identify

The parser should recognize:

1. ATX headings (`#`, `##`, `###`, etc.).
2. Paragraphs.
3. Unordered lists.
4. Ordered lists.
5. Markdown tables and separator rows.
6. Fenced code blocks using backticks or tildes.
7. Horizontal rules.
8. Blank lines and structural boundaries.

### Parsing Requirements

* Preserve the original source text and its character positions.
* Track the beginning and ending offsets of each parsed block.
* Maintain heading hierarchy through a section path.
* Do not interpret heading-like lines inside fenced code blocks as Markdown headings.
* Support both backtick and tilde code fences.
* Handle unclosed code fences safely by extending them to the end of the input.
* Preserve table rows and list item boundaries.
* Avoid modifying or stripping source content during parsing.

### Scope

This stage focuses on identifying Markdown structure.

It does not generate final chunks or perform semantic retrieval.

### Expected Outcome

A reusable parser that converts Markdown text into structured blocks with reliable source offsets, providing the foundation for structure-aware recursive chunking.


## Markdown Parser Design

#### Parser Output

Each parsed block will contain:

| Field          | Description                                 |
| -------------- | ------------------------------------------- |
| `block_type`   | Structural element type                     |
| `content`      | Original text of the block                  |
| `char_start`   | Inclusive starting character offset         |
| `char_end`     | Exclusive ending character offset           |
| `section_path` | Heading hierarchy associated with the block |
| `metadata`     | Additional structural information           |

#### Supported Block Types

* `heading`
* `paragraph`
* `unordered_list`
* `ordered_list`
* `table`
* `code`
* `horizontal_rule`

#### Parsing Strategy

The parser will process the document line by line while maintaining the original character offsets.

It will use a state-based approach to distinguish normal Markdown content from fenced code blocks.

Heading levels will be tracked using a hierarchy stack to construct the `section_path` for subsequent blocks.

#### Important Design Constraints

* Character offsets must refer to the original source string.
* Block content must not be modified or normalized.
* Code fences must support both backticks and tildes.
* Heading detection must be disabled while inside fenced code blocks.
* Unclosed code fences must extend to the end of the document.
* Consecutive list items should remain grouped logically.
* Markdown table rows should remain identifiable as table content.

#### Expected Outcome

A deterministic Markdown parser that produces structured blocks while preserving the original document content and character positions.


## Markdown Block Data Model

#### Objective

Define a dedicated data model for representing parsed Markdown blocks independently from the final `ChunkRecord` model.

#### MarkdownBlock Schema

| Field          | Type        | Description                       |
| -------------- | ----------- | --------------------------------- |
| `block_type`   | `str`       | Structural type of the block      |
| `content`      | `str`       | Original source text              |
| `char_start`   | `int`       | Inclusive starting offset         |
| `char_end`     | `int`       | Exclusive ending offset           |
| `section_path` | `list[str]` | Associated heading hierarchy      |
| `metadata`     | `dict`      | Additional structural information |

#### Design Principles

* Preserve the original source text.
* Use zero-based character offsets.
* Keep `char_end` exclusive.
* Validate that block content length matches its offsets.
* Reject unexpected fields.
* Use independent mutable defaults.

#### Separation of Responsibilities

`MarkdownBlock` represents parsed document structure.

`ChunkRecord` represents the final chunking output and its provenance.

The parser must not modify or replace the original `ChunkRecord` model.


In [14]:
from app.chunking.markdown import MarkdownBlock

sample_content = "## Introduction"

block = MarkdownBlock(
    block_type="heading",
    content=sample_content,
    char_start=0,
    char_end=len(sample_content),
    section_path=["Introduction"],
    metadata={"heading_level": 2},
)

print("MARKDOWN BLOCK MODEL")
print("=" * 50)

print("Block type:", block.block_type)
print("Content:", repr(block.content))
print("Start offset:", block.char_start)
print("End offset:", block.char_end)
print("Section path:", block.section_path)
print("Metadata:", block.metadata)

assert block.char_end - block.char_start == len(block.content)

print("\nOffset validation: PASSED")
print("Model creation: PASSED")

MARKDOWN BLOCK MODEL
Block type: heading
Content: '## Introduction'
Start offset: 0
End offset: 15
Section path: ['Introduction']
Metadata: {'heading_level': 2}

Offset validation: PASSED
Model creation: PASSED


## Markdown Block Unit Tests

#### Objective

Validate the `MarkdownBlock` model independently before implementing the Markdown parser.

#### Test Coverage

The tests will verify:

1. Valid heading block creation.
2. Valid paragraph block creation.
3. Rejection of unsupported block types.
4. Rejection of negative starting offsets.
5. Rejection of zero or negative ending offsets.
6. Rejection of inconsistent content length and offsets.
7. Rejection of unexpected fields.
8. Independence of mutable default fields.

#### Validation Criteria

All valid records must be accepted, while invalid records must raise Pydantic validation errors.

The model must preserve the original content without stripping whitespace.

### Result Interpretation

#### Test Results

All 9 MarkdownBlock model tests passed successfully.

| Test Case                             | Result |
| ------------------------------------- | ------ |
| Valid heading block                   | PASS   |
| Valid paragraph block                 | PASS   |
| Unsupported block type rejection      | PASS   |
| Negative starting offset rejection    | PASS   |
| Zero ending offset rejection          | PASS   |
| Inconsistent content length rejection | PASS   |
| Unexpected fields rejection           | PASS   |
| Independent mutable defaults          | PASS   |
| Original whitespace preservation      | PASS   |

#### Key Findings

1. The model accepts valid Markdown block records.
2. Unsupported block types are rejected.
3. Invalid character offsets are rejected.
4. Content length must match the specified offset range.
5. Unexpected fields are rejected.
6. Mutable fields are independently initialized.
7. Original whitespace is preserved.

#### Conclusion

The `MarkdownBlock` data model passed all 9 unit tests.

It provides a validated representation of Markdown structural elements while preserving the original text and character offset constraints.

**Model Validation Status: PASSED**

Note: These tests validate the data model only. Markdown parsing behavior has not yet been implemented or tested.



## Markdown Parser Implementation

### Objective

Implement a deterministic, line-based Markdown parser that identifies document structures and converts them into validated MarkdownBlock objects.

### Supported Markdown Elements

The parser will recognize the following elements:

1. ATX headings (levels 1–6).
2. Paragraphs.
3. Unordered lists.
4. Ordered lists.
5. Markdown tables.
6. Fenced code blocks using backticks or tildes.
7. Horizontal rules.

### Parsing Strategy

The parser processes the original document line by line.

Each line is classified according to Markdown structural patterns and the current parsing state.

The parser maintains a heading hierarchy to associate each block with its corresponding section path.

### Structural Parsing Rules

**Headings**

* Detect heading levels from 1 to 6.
* Extract heading text and level.
* Maintain the current heading hierarchy.
* Remove headings of the same or deeper levels when a new heading is encountered.

**Paragraphs**

* Group consecutive non-empty text lines.
* Stop when a structural boundary is encountered.
* Preserve original whitespace and line endings.

**Lists**

* Detect ordered and unordered list markers.
* Group consecutive list items.
* Preserve nested and indented continuation lines.

**Tables**

* Identify a header followed by a valid Markdown separator row.
* Group consecutive table rows.
* Preserve original table formatting.

**Code Blocks**

* Support backtick and tilde fences.
* Match closing fences using the opening fence character and minimum fence length.
* Ignore Markdown heading and list syntax inside code blocks.
* Preserve unclosed code blocks through the end of the document.

**Horizontal Rules**

* Recognize standard Markdown horizontal rule patterns.
* Represent them as independent blocks.

### Character Offset Preservation

Each block stores its original source positions:

* `char_start`: Inclusive starting offset.
* `char_end`: Exclusive ending offset.

The parser must satisfy the following invariant:

`block.content == source_text[block.char_start:block.char_end]`

### Section Hierarchy

Each block inherits the active heading hierarchy.

For example, content under a level-three heading will contain the corresponding level-one, level-two, and level-three headings in its section path.

### Determinism

Parsing identical input multiple times must produce identical block classifications, content, offsets, section paths, and metadata.

### Scope

This implementation supports common Markdown structures used in the project knowledge base. It is not intended to implement the complete CommonMark specification.

### Expected Outcome

A reusable `parse_markdown()` function that identifies structural Markdown elements while preserving their original source content and character positions.


### Markdown Line Tracking

#### Objective

Implement a line-tracking utility that identifies every line in a Markdown document while preserving its original character positions.

#### Requirements

* Track each line's original content.
* Record the inclusive starting offset.
* Record the exclusive ending offset.
* Preserve newline characters in the source boundaries.
* Support LF and CRLF line endings.
* Handle empty input and trailing newlines correctly.
* Never modify the original Markdown text.

#### Expected Outcome

A reliable line representation that provides the foundation for code fence detection, heading recognition, and structural block parsing.


In [ ]:
from app.chunking.markdown_parser import track_markdown_lines

sample_markdown = "# Introduction\nFirst paragraph.\nSecond paragraph.\r\nFinal line."

tracked_lines = track_markdown_lines(sample_markdown)

print("MARKDOWN LINE TRACKING")
print("=" * 55)

for line in tracked_lines:
    print(
        f"Line {line.line_number}: "
        f"start={line.char_start}, "
        f"end={line.char_end}, "
        f"content={line.content!r}"
    )

print("\nVALIDATION")

assert len(tracked_lines) == 4

for line in tracked_lines:
    original_segment = sample_markdown[line.char_start : line.char_end]

    assert original_segment.rstrip("\r\n") == line.content

assert tracked_lines[0].char_start == 0
assert tracked_lines[-1].char_end == len(sample_markdown)

print("Line count: PASSED")
print("Source offsets: PASSED")
print("Original content preservation: PASSED")
print("Complete source coverage: PASSED")

MARKDOWN LINE TRACKING
Line 1: start=0, end=15, content='# Introduction'
Line 2: start=15, end=32, content='First paragraph.'
Line 3: start=32, end=51, content='Second paragraph.'
Line 4: start=51, end=62, content='Final line.'

VALIDATION
Line count: PASSED
Source offsets: PASSED
Original content preservation: PASSED
Complete source coverage: PASSED


### Result Interpretation

The Markdown line-tracking experiment completed successfully.

#### Experimental Results

| Validation                         | Result |
| ---------------------------------- | ------ |
| Number of detected lines           | 4      |
| First line starts at offset 0      | PASS   |
| Source offsets are accurate        | PASS   |
| Original line content is preserved | PASS   |
| Complete source coverage           | PASS   |
| Mixed LF and CRLF handling         | PASS   |

#### Observed Line Boundaries

| Line | Start | End | Content             |
| ---- | ----: | --: | ------------------- |
| 1    |     0 |  15 | `# Introduction`    |
| 2    |    15 |  32 | `First paragraph.`  |
| 3    |    32 |  51 | `Second paragraph.` |
| 4    |    51 |  62 | `Final line.`       |

#### Key Findings

1. All four lines were identified correctly.
2. Character offsets match the original source positions.
3. Both LF and CRLF line endings were handled correctly.
4. The original line content was preserved without modifying the source.
5. The tracked ranges cover the complete input text.

#### Conclusion

The line-tracking utility successfully preserves source positions and provides a reliable foundation for Markdown structural detection.

**Validation Status: PASSED**


## Code Fence Detection

#### Objective

Implement a code fence detection utility that identifies fenced code blocks without misclassifying their internal content as Markdown elements.

#### Supported Syntax

* Backtick fences using three or more backticks.
* Tilde fences using three or more tildes.
* Optional language identifiers and information strings.
* Closing fences using the same fence character and at least the opening fence length.

#### Detection Requirements

1. Identify valid opening fences.
2. Match closing fences using the same character type.
3. Preserve the original source offsets.
4. Capture the opening fence's information string.
5. Ignore Markdown heading syntax inside code fences.
6. Support fences with different lengths.
7. Treat an unclosed fence as extending to the end of the document.

#### Expected Outcome

A deterministic detector that identifies code fence boundaries and provides the metadata required by the Markdown parser.


In [ ]:
import importlib
import app.chunking.markdown_parser as markdown_parser

importlib.reload(markdown_parser)
from app.chunking.markdown_parser import detect_code_fences

# Build the sample without embedding triple backticks directly.
backticks = "`" * 3
long_backticks = "`" * 4
tildes = "~" * 3

sample_lines = [
    "# Introduction",
    "",
    "Regular paragraph.",
    "",
    backticks + "python",
    "# This is a Python comment",
    'print("Hello")',
    "## Not a Markdown heading",
    backticks,
    "",
    tildes + "json",
    "{",
    '    "name": "Markdown Parser"',
    "}",
    tildes,
    "",
    long_backticks + "javascript",
    'console.log("Long fence");',
    long_backticks,
    "",
    tildes + "python",
    "# Unclosed code fence",
    'print("Still inside code")',
]

sample_markdown = "\n".join(sample_lines)

detected_fences = detect_code_fences(sample_markdown)

print("CODE FENCE DETECTION")
print("=" * 60)

for index, fence in enumerate(detected_fences, start=1):
    print(f"\nFence {index}")
    print("Character:", fence.fence_char)
    print("Length:", fence.fence_length)
    print("Language:", fence.info_string)
    print("Opening line:", fence.opening_line)
    print("Closing line:", fence.closing_line)
    print("Closed:", fence.closed)
    print("Offsets:", (fence.char_start, fence.char_end))

print("\nVALIDATION")

assert len(detected_fences) == 4

assert detected_fences[0].fence_char == "`"
assert detected_fences[0].info_string == "python"
assert detected_fences[0].closed is True

assert detected_fences[1].fence_char == "~"
assert detected_fences[1].info_string == "json"
assert detected_fences[1].closed is True

assert detected_fences[2].fence_length == 4
assert detected_fences[2].info_string == "javascript"
assert detected_fences[2].closed is True

assert detected_fences[3].closed is False
assert detected_fences[3].closing_line is None
assert detected_fences[3].char_end == len(sample_markdown)

for fence in detected_fences:
    original_block = sample_markdown[fence.char_start : fence.char_end]

    assert original_block.startswith(fence.fence_char * fence.fence_length)

print("Backtick fences: PASSED")
print("Tilde fences: PASSED")
print("Variable fence lengths: PASSED")
print("Language detection: PASSED")
print("Unclosed fence handling: PASSED")
print("Original source offsets: PASSED")

CODE FENCE DETECTION

Fence 1
Character: `
Length: 3
Language: python
Opening line: 5
Closing line: 9
Closed: True
Offsets: (36, 118)

Fence 2
Character: ~
Length: 3
Language: json
Opening line: 11
Closing line: 15
Closed: True
Offsets: (119, 165)

Fence 3
Character: `
Length: 4
Language: javascript
Opening line: 17
Closing line: 19
Closed: True
Offsets: (166, 213)

Fence 4
Character: ~
Length: 3
Language: python
Opening line: 21
Closing line: None
Closed: False
Offsets: (214, 272)

VALIDATION
Backtick fences: PASSED
Tilde fences: PASSED
Variable fence lengths: PASSED
Language detection: PASSED
Unclosed fence handling: PASSED
Original source offsets: PASSED


### Result Interpretation

The code fence detection experiment completed successfully.

#### Experimental Results

| Test Case                        | Detected Result         | Status |
| -------------------------------- | ----------------------- | ------ |
| Backtick Python fence            | 3 backticks, Python     | PASS   |
| Tilde JSON fence                 | 3 tildes, JSON          | PASS   |
| Variable-length JavaScript fence | 4 backticks, JavaScript | PASS   |
| Unclosed Python fence            | Detected as unclosed    | PASS   |

#### Detected Fence Boundaries

| Fence | Opening Line | Closing Line | Start Offset | End Offset | Closed |
| ----- | -----------: | -----------: | -----------: | ---------: | ------ |
| 1     |            5 |            9 |           36 |        118 | Yes    |
| 2     |           11 |           15 |          119 |        165 | Yes    |
| 3     |           17 |           19 |          166 |        213 | Yes    |
| 4     |           21 |          N/A |          214 |        272 | No     |

#### Key Findings

1. Both backtick and tilde fences are detected correctly.
2. Variable-length fences are supported.
3. Language identifiers are extracted correctly.
4. An unclosed fence extends to the end of the input.
5. Original character offsets are preserved.
6. Code fence detection is independent of the content inside each fence.

#### Conclusion

The code fence detector successfully handles all four tested scenarios.

It provides the structural boundaries needed to prevent code content from being misclassified during subsequent Markdown parsing.

**Validation Status: PASSED**

* Detected fences: 4
* Closed fences: 3
* Unclosed fences: 1
* Failed assertions: 0


## Code Fence Edge Case Testing

#### Objective

Evaluate the code fence detector against additional Markdown edge cases that may affect structural parsing.

#### Test Scenarios

| Test | Scenario                                 | Expected Behavior             |
| ---- | ---------------------------------------- | ----------------------------- |
| 1    | Fence without a language identifier      | Detect successfully           |
| 2    | Closing fence shorter than opening fence | Do not close the block        |
| 3    | Closing fence uses a different marker    | Do not close the block        |
| 4    | Heading syntax inside a code fence       | Keep it inside the code block |
| 5    | Opening fence with permitted indentation | Detect successfully           |

#### Validation Criteria

* Only matching fence characters may close an open code block.
* Closing fences must contain at least as many markers as the opening fence.
* Content inside a code block must not affect fence detection.
* Valid indentation must not prevent detection.
* Unclosed fences must extend to the end of the source.

#### Expected Outcome

Confirm that the detector handles malformed or unusual fence configurations without incorrectly terminating code blocks.


In [ ]:
from app.chunking.markdown_parser import detect_code_fences


test_cases = {
    "No language identifier": ("```\nprint('hello')\n```"),
    "Shorter closing fence": ("````python\nprint('hello')\n```\n"),
    "Different closing marker": ("~~~python\nprint('hello')\n```\n"),
    "Heading inside code fence": ("```markdown\n## This is code content\n```"),
    "Indented opening fence": ("   ```python\nprint('hello')\n   ```"),
}


print("CODE FENCE EDGE CASE TESTS")
print("=" * 60)

for name, content in test_cases.items():
    fences = detect_code_fences(content)

    print(f"\n{name}")
    print("Detected fences:", len(fences))

    for fence in fences:
        print(
            f"  Opening line: {fence.opening_line}, "
            f"Closing line: {fence.closing_line}, "
            f"Closed: {fence.closed}, "
            f"Language: {fence.info_string!r}"
        )


# Assertions

assert len(detect_code_fences(test_cases["No language identifier"])) == 1

shorter_fence_result = detect_code_fences(test_cases["Shorter closing fence"])
assert len(shorter_fence_result) == 1
assert shorter_fence_result[0].closed is False

different_marker_result = detect_code_fences(test_cases["Different closing marker"])
assert len(different_marker_result) == 1
assert different_marker_result[0].closed is False

heading_inside_result = detect_code_fences(test_cases["Heading inside code fence"])
assert len(heading_inside_result) == 1
assert heading_inside_result[0].closed is True

indented_result = detect_code_fences(test_cases["Indented opening fence"])
assert len(indented_result) == 1
assert indented_result[0].closed is True

print("\nALL EDGE CASE ASSERTIONS PASSED")

CODE FENCE EDGE CASE TESTS

No language identifier
Detected fences: 1
  Opening line: 1, Closing line: 3, Closed: True, Language: ''

Shorter closing fence
Detected fences: 1
  Opening line: 1, Closing line: None, Closed: False, Language: 'python'

Different closing marker
Detected fences: 1
  Opening line: 1, Closing line: None, Closed: False, Language: 'python'

Heading inside code fence
Detected fences: 1
  Opening line: 1, Closing line: 3, Closed: True, Language: 'markdown'

Indented opening fence
Detected fences: 1
  Opening line: 1, Closing line: 3, Closed: True, Language: 'python'

ALL EDGE CASE ASSERTIONS PASSED


### Result Interpretation

All five code fence edge case tests passed successfully.

#### Test Results

| Test Scenario             | Actual Behavior                                 | Status |
| ------------------------- | ----------------------------------------------- | ------ |
| No language identifier    | Fence detected with an empty info string        | PASS   |
| Shorter closing fence     | Block remained unclosed                         | PASS   |
| Different closing marker  | Block remained unclosed                         | PASS   |
| Heading inside code fence | Content remained inside the detected code block | PASS   |
| Indented opening fence    | Fence detected successfully                     | PASS   |

#### Key Findings

1. Language identifiers are optional.
2. A closing fence shorter than the opening fence cannot terminate the block.
3. A closing fence using a different marker character cannot terminate the block.
4. Markdown-like content inside code fences does not interfere with fence detection.
5. Opening fences with up to three leading spaces are supported.

#### Conclusion

The detector passed all five additional edge case tests.

Together with the previous experiment, all nine tested code fence scenarios passed successfully.

**Validation Status: PASSED**


## Heading Detection and Section Hierarchy

#### Objective

Implement heading recognition and hierarchical section tracking for Markdown documents.

#### Supported Syntax

This stage supports ATX-style Markdown headings:

* `# Heading`
* `## Heading`
* `### Heading`
* Up to six heading levels.

#### Heading Detection Requirements

1. Recognize headings with one to six leading hash characters.
2. Allow up to three leading spaces.
3. Require whitespace or the end of the line after the heading markers.
4. Extract heading text without modifying the original source content.
5. Ignore heading-like lines inside fenced code blocks.
6. Track the active heading hierarchy.
7. Remove lower-level headings when a higher-level heading is encountered.
8. Preserve heading levels in block metadata.

#### Section Path

Each heading and subsequent content block must be associated with the active heading hierarchy.

For example, a level-three heading must inherit its level-one and level-two parent headings when available.

#### Expected Outcome

A deterministic heading detector that builds reliable section paths and provides structural context for subsequent Markdown block classification.


In [ ]:
import importlib
import app.chunking.markdown_parser as markdown_parser

importlib.reload(markdown_parser)
from app.chunking.markdown_parser import detect_headings

sample_markdown = "\n".join(
    [
        "# Machine Learning",
        "",
        "## Supervised Learning",
        "",
        "### Classification",
        "",
        "### Regression",
        "",
        "## Unsupervised Learning",
        "",
        "#### Clustering",
        "",
        "   # Indented Heading",
        "",
        "####### Invalid Heading",
        "",
        "```markdown",
        "# This is not a heading",
        "## Neither is this",
        "```",
        "",
        "## Final Section",
    ]
)

detected_headings = detect_headings(sample_markdown)

print("HEADING DETECTION")
print("=" * 60)

for heading in detected_headings:
    print(
        f"Level {heading.metadata['heading_level']}: {heading.metadata['heading_text']}"
    )
    print("Section path:", heading.section_path)
    print("Offsets:", (heading.char_start, heading.char_end))
    print()

print("VALIDATION")

assert len(detected_headings) == 8

assert detected_headings[0].section_path == ["Machine Learning"]

assert detected_headings[2].section_path == [
    "Machine Learning",
    "Supervised Learning",
    "Classification",
]

assert detected_headings[4].section_path == [
    "Machine Learning",
    "Unsupervised Learning",
]

assert detected_headings[5].section_path == [
    "Machine Learning",
    "Unsupervised Learning",
    "Clustering",
]

assert detected_headings[-1].section_path == [
    "Indented Heading",
    "Final Section",
]

assert all(heading.char_end > heading.char_start for heading in detected_headings)

print("Heading levels: PASSED")
print("Section hierarchy: PASSED")
print("Indented headings: PASSED")
print("Invalid heading rejection: PASSED")
print("Code fence exclusion: PASSED")
print("Source offsets: PASSED")

HEADING DETECTION
Level 1: Machine Learning
Section path: ['Machine Learning']
Offsets: (0, 18)

Level 2: Supervised Learning
Section path: ['Machine Learning', 'Supervised Learning']
Offsets: (20, 42)

Level 3: Classification
Section path: ['Machine Learning', 'Supervised Learning', 'Classification']
Offsets: (44, 62)

Level 3: Regression
Section path: ['Machine Learning', 'Supervised Learning', 'Regression']
Offsets: (64, 78)

Level 2: Unsupervised Learning
Section path: ['Machine Learning', 'Unsupervised Learning']
Offsets: (80, 104)

Level 4: Clustering
Section path: ['Machine Learning', 'Unsupervised Learning', 'Clustering']
Offsets: (106, 121)

Level 1: Indented Heading
Section path: ['Indented Heading']
Offsets: (123, 144)

Level 2: Final Section
Section path: ['Indented Heading', 'Final Section']
Offsets: (231, 247)

VALIDATION
Heading levels: PASSED
Section hierarchy: PASSED
Indented headings: PASSED
Invalid heading rejection: PASSED
Code fence exclusion: PASSED
Source offsets

### Results and Interpretation

The heading detection experiment completed successfully.

**Observed Results:**

* Detected 8 valid ATX headings.
* Correctly recognized heading levels from 1 to 4 in the test input.
* Maintained hierarchical section paths across heading-level transitions.
* Recognized headings with up to three leading spaces.
* Rejected headings containing more than six hash characters.
* Ignored heading-like lines inside fenced code blocks.
* Preserved the original character offsets of detected headings.

**Important Observation:**

The indented heading `Indented Heading` was correctly recognized as a level-one heading. Consequently, `Final Section` inherited it as its parent section.

This confirms that the parser follows the configured ATX heading rules rather than assuming that every new heading belongs to the first document section.

**Validation Summary:**

All heading detection assertions passed.

The heading detector is ready for integration into the broader Markdown block classification process.


## Heading Detection Edge Cases

#### Objective

Evaluate heading recognition under boundary conditions and less common Markdown heading patterns.

#### Test Scenarios

1. Empty headings must be rejected.
2. Whitespace-only headings must be rejected.
3. Optional closing hash sequences must be handled correctly.
4. Missing whitespace after heading markers must be rejected.
5. Six-level headings must be accepted.
6. Seven-level headings must be rejected.
7. Arabic heading text must be preserved.
8. Inline Markdown formatting must remain unchanged in the original heading content.

#### Expected Outcome

Confirm that heading detection handles valid and invalid ATX syntax consistently while preserving Unicode text and source offsets.


In [28]:
from app.chunking.markdown_parser import detect_headings

test_cases = {
    "Empty heading": "#",
    "Whitespace-only heading": "#   ",
    "Closing hashes": "## Python ##",
    "Missing whitespace": "#Python",
    "Six-level heading": "###### Deep Topic",
    "Seven-level heading": "####### Invalid Topic",
    "Arabic heading": "## هندسة الذكاء الاصطناعي",
    "Inline formatting": "### Learning **Deep Learning**",
}

print("HEADING EDGE CASE TESTS")
print("=" * 65)

for name, content in test_cases.items():
    headings = detect_headings(content)

    print(f"\n{name}")
    print("Input:", repr(content))
    print("Detected:", len(headings))

    if headings:
        heading = headings[0]
        print("Heading text:", repr(heading.metadata["heading_text"]))
        print("Level:", heading.metadata["heading_level"])
        print("Original content:", repr(heading.content))
        print("Offsets:", (heading.char_start, heading.char_end))

HEADING EDGE CASE TESTS

Empty heading
Input: '#'
Detected: 0

Whitespace-only heading
Input: '#   '
Detected: 0

Closing hashes
Input: '## Python ##'
Detected: 1
Heading text: 'Python'
Level: 2
Original content: '## Python ##'
Offsets: (0, 12)

Missing whitespace
Input: '#Python'
Detected: 0

Six-level heading
Input: '###### Deep Topic'
Detected: 1
Heading text: 'Deep Topic'
Level: 6
Original content: '###### Deep Topic'
Offsets: (0, 17)

Seven-level heading
Input: '####### Invalid Topic'
Detected: 0

Arabic heading
Input: '## هندسة الذكاء الاصطناعي'
Detected: 1
Heading text: 'هندسة الذكاء الاصطناعي'
Level: 2
Original content: '## هندسة الذكاء الاصطناعي'
Offsets: (0, 25)

Inline formatting
Input: '### Learning **Deep Learning**'
Detected: 1
Heading text: 'Learning **Deep Learning**'
Level: 3
Original content: '### Learning **Deep Learning**'
Offsets: (0, 30)


In [ ]:
# Empty headings must be rejected.
assert detect_headings(test_cases["Empty heading"]) == []

# Whitespace-only headings must be rejected.
assert detect_headings(test_cases["Whitespace-only heading"]) == []

# Closing hashes must be excluded from heading text.
closing_hashes = detect_headings(test_cases["Closing hashes"])
assert len(closing_hashes) == 1
assert closing_hashes[0].metadata["heading_text"] == "Python"

# Missing whitespace after the heading marker must be rejected.
assert detect_headings(test_cases["Missing whitespace"]) == []

# Six heading levels must be accepted.
six_level = detect_headings(test_cases["Six-level heading"])
assert len(six_level) == 1
assert six_level[0].metadata["heading_level"] == 6

# Seven heading levels must be rejected.
assert detect_headings(test_cases["Seven-level heading"]) == []

# Arabic heading text must be preserved.
arabic = detect_headings(test_cases["Arabic heading"])
assert len(arabic) == 1
assert arabic[0].metadata["heading_text"] == "هندسة الذكاء الاصطناعي"

# Inline formatting must remain unchanged in the original content.
formatted = detect_headings(test_cases["Inline formatting"])
assert len(formatted) == 1
assert formatted[0].content == "### Learning **Deep Learning**"
assert formatted[0].metadata["heading_text"] == "Learning **Deep Learning**"

# Validate original source offsets for all detected headings.
for content in test_cases.values():
    for heading in detect_headings(content):
        assert heading.content == content[heading.char_start : heading.char_end]

print("All heading edge-case assertions passed.")

All heading edge-case assertions passed.


### Result Interpretation

All eight heading edge-case scenarios passed successfully.

Key findings:

1. Empty and whitespace-only headings were correctly rejected.
2. Optional closing hash sequences were handled correctly.
3. Headings without required whitespace after the hash markers were rejected.
4. Six-level headings were accepted, while seven-level headings were rejected.
5. Arabic heading text was preserved correctly.
6. Inline Markdown formatting remained unchanged in the original heading content.
7. Character offsets correctly identified the original heading text in the source.

### Conclusion

The heading detector handles the tested ATX heading edge cases reliably. It preserves Unicode content, Markdown formatting, and source offsets, providing a validated foundation for subsequent Markdown block classification.


## Markdown Block Classification

#### Objective

Implement structural classification for Markdown content beyond heading detection.

#### Supported Block Types

* Paragraph
* Unordered List
* Ordered List
* Table
* Code Block
* Horizontal Rule

#### Classification Requirements

1. Preserve the original source order.
2. Preserve exact source character offsets.
3. Associate content blocks with the active heading hierarchy.
4. Prevent code-fenced content from being misclassified.
5. Avoid overlapping source ranges between independent blocks.
6. Preserve the original Markdown text.

#### Implementation Strategy

Classification will be implemented incrementally, starting with paragraphs and then extending to lists, tables, code blocks, and horizontal rules.

#### Expected Outcome

A deterministic Markdown block classifier that produces structurally meaningful blocks while preserving source provenance.


### Paragraph Detection

### Objective

Detect paragraph blocks in Markdown documents while preserving their original content and source positions.

### Requirements

1. Group consecutive non-empty text lines into paragraphs.
2. Separate paragraphs using blank lines.
3. Exclude headings and fenced code blocks.
4. Exclude recognizable list items and horizontal rules.
5. Preserve the original source text.
6. Maintain accurate character offsets.
7. Associate each paragraph with its active section hierarchy.
8. Preserve the original document order.

### Expected Outcome

A deterministic paragraph detector that returns validated MarkdownBlock objects with accurate source provenance.


In [ ]:
import importlib
import app.chunking.markdown_parser as markdown_parser

importlib.reload(markdown_parser)
from app.chunking.markdown_parser import detect_paragraphs

# Build a Markdown sample without nested quotation delimiters.
paragraph_sample = "\n".join(
    [
        "# Machine Learning",
        "",
        "Machine learning enables systems to learn from data.",
        "It identifies patterns and supports predictions.",
        "",
        "## Deep Learning",
        "",
        "Deep learning uses neural networks.",
        "",
        "```python",
        "model.fit(X, y)",
        "```",
        "",
        "- Supervised Learning",
        "- Unsupervised Learning",
        "",
        "This is the final paragraph.",
    ]
)

# Detect paragraphs.
paragraphs = detect_paragraphs(paragraph_sample)

# Display results.
print("PARAGRAPH DETECTION")
print("=" * 65)

for index, paragraph in enumerate(paragraphs, start=1):
    print(f"\nParagraph {index}")
    print("Content:", repr(paragraph.content))
    print("Section:", paragraph.section_path)
    print("Offsets:", (paragraph.char_start, paragraph.char_end))

# Validate paragraph count.
assert len(paragraphs) == 3

# Validate section hierarchy.
assert paragraphs[0].section_path == ["Machine Learning"]

assert paragraphs[1].section_path == [
    "Machine Learning",
    "Deep Learning",
]

assert paragraphs[2].section_path == [
    "Machine Learning",
    "Deep Learning",
]

# Ensure code blocks are excluded.
assert "model.fit" not in "\n".join(paragraph.content for paragraph in paragraphs)

# Ensure list items are excluded.
assert "Supervised Learning" not in "\n".join(
    paragraph.content for paragraph in paragraphs
)

# Validate source content and offsets.
for paragraph in paragraphs:
    assert (
        paragraph.content == paragraph_sample[paragraph.char_start : paragraph.char_end]
    )

    assert paragraph.block_type == "paragraph"

print("\n" + "=" * 65)
print("ALL PARAGRAPH DETECTION ASSERTIONS PASSED")

PARAGRAPH DETECTION

Paragraph 1
Content: 'Machine learning enables systems to learn from data.\nIt identifies patterns and supports predictions.'
Section: ['Machine Learning']
Offsets: (20, 121)

Paragraph 2
Content: 'Deep learning uses neural networks.'
Section: ['Machine Learning', 'Deep Learning']
Offsets: (141, 176)

Paragraph 3
Content: 'This is the final paragraph.'
Section: ['Machine Learning', 'Deep Learning']
Offsets: (256, 284)

ALL PARAGRAPH DETECTION ASSERTIONS PASSED


#### Result Interpretation

Paragraph detection completed successfully.

Key findings:

1. Three paragraph blocks were detected correctly.
2. Consecutive non-empty text lines were grouped into a single paragraph.
3. Paragraphs were separated correctly by blank lines.
4. Heading blocks were excluded from paragraph classification.
5. Fenced code blocks and unordered list items were excluded.
6. Section paths were preserved according to the active heading hierarchy.
7. Source character offsets matched the original paragraph content exactly.

### Conclusion

The paragraph detector passed all implemented validation checks. It preserves source provenance and heading context while correctly separating paragraph content from the tested Markdown structures.


## Unordered List Detection

### Objective

Implement a detector for unordered Markdown lists and represent consecutive list items as structured `MarkdownBlock` objects.

### Supported Syntax

The detector supports the following unordered list markers:

* Hyphen (`-`)
* Asterisk (`*`)
* Plus (`+`)

### Detection Requirements

1. Recognize list items with up to three leading spaces.
2. Require whitespace after the list marker.
3. Group consecutive unordered list items into a single block.
4. Separate independent lists when blank lines or other block types occur.
5. Ignore list-like lines inside fenced code blocks.
6. Preserve the original Markdown content without modification.
7. Maintain accurate character offsets.
8. Preserve the active heading hierarchy through `section_path`.
9. Return blocks in their original document order.

### Expected Outcome

A deterministic unordered list detector that identifies consecutive list items while preserving source content, character offsets, and document section context.

### Validation Criteria

* Correct list block count.
* Correct grouping of consecutive list items.
* Correct section hierarchy.
* Exclusion of code-fenced content.
* Exact source substring matching.


In [ ]:
import importlib
import app.chunking.markdown_parser as markdown_parser

importlib.reload(markdown_parser)
from app.chunking.markdown_parser import detect_unordered_lists

list_sample = "\n".join(
    [
        "# Machine Learning",
        "",
        "Introduction to machine learning.",
        "",
        "- Supervised Learning",
        "- Unsupervised Learning",
        "- Reinforcement Learning",
        "",
        "## Deep Learning",
        "",
        "- Neural Networks",
        "- Transformers",
        "",
        "```python",
        "- This is code, not a list",
        "```",
        "",
        "Final paragraph.",
    ]
)

# Detect unordered lists.
list_blocks = detect_unordered_lists(list_sample)

# Display results.
print("UNORDERED LIST DETECTION")
print("=" * 65)

for index, block in enumerate(list_blocks, start=1):
    print(f"\nList {index}")
    print("Content:", repr(block.content))
    print("Section:", block.section_path)
    print("Offsets:", (block.char_start, block.char_end))

# Validate results.
assert len(list_blocks) == 2

assert list_blocks[0].section_path == ["Machine Learning"]

assert list_blocks[1].section_path == [
    "Machine Learning",
    "Deep Learning",
]

assert "This is code" not in "\n".join(block.content for block in list_blocks)

assert all(block.block_type == "unordered_list" for block in list_blocks)

for block in list_blocks:
    assert block.content == list_sample[block.char_start : block.char_end]

print("\n" + "=" * 65)
print("ALL UNORDERED LIST ASSERTIONS PASSED")

UNORDERED LIST DETECTION

List 1
Content: '- Supervised Learning\n- Unsupervised Learning\n- Reinforcement Learning'
Section: ['Machine Learning']
Offsets: (55, 125)

List 2
Content: '- Neural Networks\n- Transformers'
Section: ['Machine Learning', 'Deep Learning']
Offsets: (145, 177)

ALL UNORDERED LIST ASSERTIONS PASSED


### Result Interpretation

Unordered list detection completed successfully.

Key findings:

1. Two independent unordered list blocks were detected.
2. Consecutive list items were grouped correctly.
3. The first list contained three items.
4. The second list contained two items.
5. Both lists inherited the correct section hierarchy.
6. List-like content inside fenced code blocks was excluded.
7. Character offsets matched the original source content exactly.
8. All implemented assertions passed.

### Conclusion

The unordered list detector successfully identifies and groups consecutive Markdown list items while preserving source content, character offsets, and section context.


## Ordered List Detection

### Objective

Implement a detector for ordered Markdown lists and represent consecutive ordered list items as structured MarkdownBlock objects.

### Supported Syntax

* Number followed by a period: `1.`
* Number followed by a closing parenthesis: `1)`

### Detection Requirements

1. Recognize ordered list markers with up to three leading spaces.
2. Require whitespace after the list marker.
3. Group consecutive ordered list items into a single block.
4. Separate independent lists when blank lines or other block types occur.
5. Ignore ordered list-like content inside fenced code blocks.
6. Preserve the original Markdown content.
7. Maintain accurate character offsets.
8. Preserve the active section hierarchy.
9. Return blocks in their original document order.

### Expected Outcome

A deterministic ordered list detector that preserves source content, provenance, and section context.


In [ ]:
# Test ordered list detection.
ordered_sample = "\n".join(
    [
        "# Machine Learning",
        "",
        "1. Data Collection",
        "2. Data Cleaning",
        "3. Model Training",
        "",
        "## Deep Learning",
        "",
        "1) Neural Networks",
        "2) Transformers",
        "",
        "```python",
        "1. This is code, not a list",
        "```",
        "",
        "Final paragraph.",
    ]
)

ordered_blocks = detect_ordered_lists(ordered_sample)

print("ORDERED LIST DETECTION")
print("=" * 65)

for index, block in enumerate(ordered_blocks, start=1):
    print(f"\nList {index}")
    print("Content:", repr(block.content))
    print("Section:", block.section_path)
    print("Offsets:", (block.char_start, block.char_end))

# Validate results.
assert len(ordered_blocks) == 2

assert ordered_blocks[0].section_path == ["Machine Learning"]

assert ordered_blocks[1].section_path == [
    "Machine Learning",
    "Deep Learning",
]

assert "This is code" not in "\n".join(block.content for block in ordered_blocks)

assert all(block.block_type == "ordered_list" for block in ordered_blocks)

for block in ordered_blocks:
    assert block.content == ordered_sample[block.char_start : block.char_end]

print("\n" + "=" * 65)
print("ALL ORDERED LIST ASSERTIONS PASSED")

ORDERED LIST DETECTION

List 1
Content: '1. Data Collection\n2. Data Cleaning\n3. Model Training'
Section: ['Machine Learning']
Offsets: (20, 73)

List 2
Content: '1) Neural Networks\n2) Transformers'
Section: ['Machine Learning', 'Deep Learning']
Offsets: (93, 127)

ALL ORDERED LIST ASSERTIONS PASSED


### Result Interpretation

Ordered list detection completed successfully.

Key findings:

1. Two independent ordered list blocks were detected.
2. The first list contained three numbered items using period markers (`1.`, `2.`, `3.`).
3. The second list contained two numbered items using parenthesis markers (`1)`, `2)`).
4. Consecutive list items were grouped correctly.
5. Both lists inherited the correct section hierarchy.
6. Ordered list-like content inside fenced code blocks was excluded.
7. Character offsets matched the original source content exactly.
8. All implemented assertions passed.

### Conclusion

The ordered list detector correctly identifies and groups consecutive numbered Markdown list items while preserving source content, character offsets, and section context.


## Markdown Table Detection

### Objective

Implement a detector for Markdown tables and represent each complete table as a structured MarkdownBlock.

### Supported Syntax

GitHub Flavored Markdown (GFM) pipe tables:

* Header rows containing pipe separators.
* Separator rows containing at least three hyphens per column.
* Optional column alignment markers using colons.
* Subsequent data rows.

### Detection Requirements

1. Identify a header row followed by a valid separator row.
2. Validate separator cells.
3. Support left, center, and right alignment markers.
4. Group consecutive table rows into one block.
5. Stop table detection when a non-table block is encountered.
6. Ignore table-like content inside fenced code blocks.
7. Preserve original Markdown content and whitespace.
8. Maintain accurate character offsets.
9. Preserve the active section hierarchy.

### Expected Outcome

A deterministic Markdown table detector that identifies complete GFM pipe tables while preserving source content and provenance.


In [ ]:
import importlib
import app.chunking.markdown_parser as markdown_parser

importlib.reload(markdown_parser)
from app.chunking.markdown_parser import detect_markdown_tables

table_sample = "\n".join(
    [
        "# Machine Learning",
        "",
        "| Algorithm | Type | Accuracy |",
        "|:----------|:----:|---------:|",
        "| Random Forest | Ensemble | 95% |",
        "| SVM | Supervised | 92% |",
        "",
        "## Deep Learning",
        "",
        "| Model | Architecture |",
        "|-------|--------------|",
        "| CNN | Convolutional |",
        "| Transformer | Attention |",
        "",
        "```markdown",
        "| Fake | Table |",
        "|------|-------|",
        "| A | B |",
        "```",
        "",
        "Final paragraph.",
    ]
)

table_blocks = detect_markdown_tables(table_sample)

print("MARKDOWN TABLE DETECTION")
print("=" * 65)

for index, table in enumerate(table_blocks, start=1):
    print(f"\nTable {index}")
    print("Content:\n", table.content)
    print("Section:", table.section_path)
    print("Columns:", table.metadata["column_count"])
    print("Header:", table.metadata["header"])
    print("Offsets:", (table.char_start, table.char_end))

# Validate table count.
assert len(table_blocks) == 2

# Validate section hierarchy.
assert table_blocks[0].section_path == ["Machine Learning"]

assert table_blocks[1].section_path == [
    "Machine Learning",
    "Deep Learning",
]

# Validate column counts.
assert table_blocks[0].metadata["column_count"] == 3
assert table_blocks[1].metadata["column_count"] == 2

# Validate headers.
assert table_blocks[0].metadata["header"] == [
    "Algorithm",
    "Type",
    "Accuracy",
]

assert table_blocks[1].metadata["header"] == [
    "Model",
    "Architecture",
]

# Ensure code-fenced table content is excluded.
assert "Fake" not in "\n".join(table.content for table in table_blocks)

# Validate exact source content and offsets.
for table in table_blocks:
    assert table.block_type == "table"

    assert table.content == table_sample[table.char_start : table.char_end]

print("\n" + "=" * 65)
print("ALL MARKDOWN TABLE ASSERTIONS PASSED")

MARKDOWN TABLE DETECTION

Table 1
Content:
 | Algorithm | Type | Accuracy |
|:----------|:----:|---------:|
| Random Forest | Ensemble | 95% |
| SVM | Supervised | 92% |
Section: ['Machine Learning']
Columns: 3
Header: ['Algorithm', 'Type', 'Accuracy']
Offsets: (20, 145)

Table 2
Content:
 | Model | Architecture |
|-------|--------------|
| CNN | Convolutional |
| Transformer | Attention |
Section: ['Machine Learning', 'Deep Learning']
Columns: 2
Header: ['Model', 'Architecture']
Offsets: (165, 266)

ALL MARKDOWN TABLE ASSERTIONS PASSED


### Result Interpretation

Markdown table detection completed successfully.

Key findings:

1. Two independent GFM pipe tables were detected.
2. The first table contained three columns and two data rows.
3. The second table contained two columns and two data rows.
4. Header cells were extracted correctly.
5. Alignment separator syntax was recognized.
6. Both tables inherited the correct section hierarchy.
7. Table-like content inside fenced code blocks was excluded.
8. Character offsets matched the original source content exactly.
9. All implemented assertions passed.

### Conclusion

The Markdown table detector successfully identifies the tested GFM pipe tables while preserving their original content, structural metadata, character offsets, and section context.


## Code Block Classification

### Objective

Convert detected Markdown fenced code regions into structured MarkdownBlock objects.

### Detection Requirements

1. Reuse the existing code fence detection implementation.
2. Support backtick and tilde fences.
3. Preserve the complete original code block, including opening and closing fences.
4. Extract the programming language or information string.
5. Preserve accurate source character offsets.
6. Associate each code block with its active section hierarchy.
7. Handle unclosed code fences safely.
8. Preserve document order.

### Expected Outcome

A deterministic code block classifier that converts detected fenced regions into MarkdownBlock objects while preserving source provenance and language metadata.


In [ ]:
import importlib
import app.chunking.markdown_parser as markdown_parser

importlib.reload(markdown_parser)
from app.chunking.markdown_parser import detect_code_blocks

code_sample = "\n".join(
    [
        "# Machine Learning",
        "",
        "Python example:",
        "",
        "```python",
        "model.fit(X, y)",
        "```",
        "",
        "## Data Processing",
        "",
        "~~~json",
        '{"status": "success"}',
        "~~~",
        "",
        "## Unclosed Example",
        "",
        "```javascript",
        "console.log('Hello');",
    ]
)

code_blocks = detect_code_blocks(code_sample)

print("CODE BLOCK CLASSIFICATION")
print("=" * 65)

for index, block in enumerate(code_blocks, start=1):
    print(f"\nCode Block {index}")
    print("Content:", repr(block.content))
    print("Section:", block.section_path)
    print("Language:", block.metadata["language"])
    print("Fence:", block.metadata["fence_char"])
    print("Closed:", block.metadata["closed"])
    print("Offsets:", (block.char_start, block.char_end))

# Validate block count.
assert len(code_blocks) == 3

# Validate programming languages.
assert [block.metadata["language"] for block in code_blocks] == [
    "python",
    "json",
    "javascript",
]

# Validate section hierarchy.
assert code_blocks[0].section_path == ["Machine Learning"]

assert code_blocks[1].section_path == [
    "Machine Learning",
    "Data Processing",
]

assert code_blocks[2].section_path == [
    "Machine Learning",
    "Unclosed Example",
]

# Validate fence types.
assert code_blocks[0].metadata["fence_char"] == "`"
assert code_blocks[1].metadata["fence_char"] == "~"

# Validate closed and unclosed fences.
assert code_blocks[0].metadata["closed"] is True
assert code_blocks[1].metadata["closed"] is True
assert code_blocks[2].metadata["closed"] is False

# Validate source content and offsets.
for block in code_blocks:
    assert block.block_type == "code"

    assert block.content == code_sample[block.char_start : block.char_end]

print("\n" + "=" * 65)
print("ALL CODE BLOCK ASSERTIONS PASSED")

CODE BLOCK CLASSIFICATION

Code Block 1
Content: '```python\nmodel.fit(X, y)\n```\n'
Section: ['Machine Learning']
Language: python
Fence: `
Closed: True
Offsets: (37, 67)

Code Block 2
Content: '~~~json\n{"status": "success"}\n~~~\n'
Section: ['Machine Learning', 'Data Processing']
Language: json
Fence: ~
Closed: True
Offsets: (88, 122)

Code Block 3
Content: "```javascript\nconsole.log('Hello');"
Section: ['Machine Learning', 'Unclosed Example']
Language: javascript
Fence: `
Closed: False
Offsets: (144, 179)

ALL CODE BLOCK ASSERTIONS PASSED


### Result Interpretation

Code block classification completed successfully.

Key findings:

1. Three fenced code blocks were detected.
2. Backtick and tilde fence styles were both recognized.
3. Programming language metadata was extracted correctly for Python, JSON, and JavaScript.
4. The first two code blocks were identified as properly closed.
5. The final JavaScript code block was correctly identified as unclosed.
6. Each code block inherited the correct section hierarchy.
7. Original source content and character offsets were preserved.
8. All implemented assertions passed.

### Conclusion

The code block classifier successfully converts detected fenced regions into structured MarkdownBlock objects while preserving language metadata, source provenance, and section context.


## Horizontal Rule Detection

### Objective

Implement a detector for Markdown horizontal rules and represent them as structured MarkdownBlock objects.

### Supported Syntax

* Three or more hyphens (`-`)
* Three or more asterisks (`*`)
* Three or more underscores (`_`)

### Detection Requirements

1. Support up to three leading spaces.
2. Allow optional spaces between rule characters.
3. Require at least three identical rule characters.
4. Avoid classifying list items as horizontal rules.
5. Ignore horizontal-rule-like content inside fenced code blocks.
6. Preserve the original source content.
7. Maintain accurate character offsets.
8. Preserve the active section hierarchy.
9. Return blocks in document order.

### Expected Outcome

A deterministic horizontal rule detector that identifies valid Markdown thematic breaks while preserving source provenance and section context.


In [ ]:
import importlib
import app.chunking.markdown_parser as markdown_parser

importlib.reload(markdown_parser)
from app.chunking.markdown_parser import detect_horizontal_rules

rule_sample = "\n".join(
    [
        "# Machine Learning",
        "",
        "---",
        "",
        "Introduction.",
        "",
        "***",
        "",
        "## Deep Learning",
        "",
        "___",
        "",
        "   -",
        "",
        "```markdown",
        "---",
        "***",
        "```",
        "",
        "Final paragraph.",
    ]
)

rules = detect_horizontal_rules(rule_sample)

print("HORIZONTAL RULE DETECTION")
print("=" * 65)

for index, rule in enumerate(rules, start=1):
    print(f"\nRule {index}")
    print("Content:", repr(rule.content))
    print("Section:", rule.section_path)
    print("Offsets:", (rule.char_start, rule.char_end))

# Validate the number of detected rules.
assert len(rules) == 3

# Validate supported rule syntax.
assert [rule.content for rule in rules] == [
    "---",
    "***",
    "___",
]

# Validate section hierarchy.
assert rules[0].section_path == ["Machine Learning"]
assert rules[1].section_path == ["Machine Learning"]
assert rules[2].section_path == [
    "Machine Learning",
    "Deep Learning",
]

# Ensure code-fenced rules are excluded.
assert all(
    rule.content not in ("---", "***")
    or rule.char_start < rule_sample.index("```markdown")
    for rule in rules
)

# Validate source content and offsets.
for rule in rules:
    assert rule.block_type == "horizontal_rule"

    assert rule.content == rule_sample[rule.char_start : rule.char_end]

print("\n" + "=" * 65)
print("ALL HORIZONTAL RULE ASSERTIONS PASSED")

HORIZONTAL RULE DETECTION

Rule 1
Content: '---'
Section: ['Machine Learning']
Offsets: (20, 23)

Rule 2
Content: '***'
Section: ['Machine Learning']
Offsets: (40, 43)

Rule 3
Content: '___'
Section: ['Machine Learning', 'Deep Learning']
Offsets: (63, 66)

ALL HORIZONTAL RULE ASSERTIONS PASSED


### Result Interpretation

The horizontal rule detector successfully identified all three supported Markdown thematic break patterns: `---`, `***`, and `___`.

The experiment confirmed that:

* All three valid horizontal rules were detected.
* Each rule preserved its exact source content and character offsets.
* Section paths correctly reflected the active heading hierarchy.
* Horizontal-rule-like content inside fenced code blocks was excluded.
* All validation assertions passed.

**Conclusion:** Horizontal rule detection is working as expected and preserves source provenance and section context.


## Markdown Block Parser Integration

#### Objective

Integrate the individual Markdown structure detectors into a unified parsing function.

#### Integrated Block Types

* Headings
* Paragraphs
* Unordered lists
* Ordered lists
* Tables
* Code blocks
* Horizontal rules

#### Integration Requirements

1. Execute all individual structure detectors.
2. Combine their results into a single collection.
3. Sort blocks by their original source positions.
4. Preserve each block's character offsets and section path.
5. Verify that detected blocks retain their original source content.
6. Identify overlapping blocks for further parser refinement.

#### Expected Outcome

A unified Markdown parsing interface that produces an ordered collection of structurally classified blocks while preserving source provenance.


In [ ]:
from app.chunking.markdown_parser import parse_markdown_blocks

markdown_sample = "\n".join(
    [
        "# Machine Learning",
        "",
        "Machine learning enables systems to learn from data.",
        "It identifies patterns and supports predictions.",
        "",
        "## Learning Algorithms",
        "",
        "- Supervised Learning",
        "- Unsupervised Learning",
        "",
        "### Model Comparison",
        "",
        "| Algorithm | Type |",
        "|---|---|",
        "| Linear Regression | Supervised |",
        "| K-Means | Unsupervised |",
        "",
        "```python",
        "# This is a code comment, not a heading.",
        "print('Hello AI')",
        "---",
        "```",
        "",
        "## Deep Learning",
        "",
        "1. Neural Networks",
        "2. Transformers",
        "",
        "Deep learning uses multiple neural network layers.",
        "",
        "---",
        "",
        "Final section of the document.",
    ]
)

blocks = parse_markdown_blocks(markdown_sample)

print("UNIFIED MARKDOWN PARSER")
print("=" * 75)

for index, block in enumerate(blocks, start=1):
    print(f"\nBlock {index}")
    print(f"Type: {block.block_type}")
    print(f"Content: {block.content!r}")
    print(f"Section: {block.section_path}")
    print(f"Offsets: ({block.char_start}, {block.char_end})")

expected_types = {
    "heading",
    "paragraph",
    "unordered_list",
    "ordered_list",
    "table",
    "code",
    "horizontal_rule",
}

detected_types = {block.block_type for block in blocks}

assert expected_types.issubset(detected_types)

for current, following in zip(blocks, blocks[1:]):
    assert current.char_end <= following.char_start

for block in blocks:
    assert markdown_sample[block.char_start : block.char_end] == block.content

print("\n" + "=" * 75)
print(f"TOTAL BLOCKS: {len(blocks)}")
print(f"DETECTED TYPES: {sorted(detected_types)}")
print("ALL UNIFIED PARSER ASSERTIONS PASSED")

UNIFIED MARKDOWN PARSER

Block 1
Type: heading
Content: '# Machine Learning'
Section: ['Machine Learning']
Offsets: (0, 18)

Block 2
Type: paragraph
Content: 'Machine learning enables systems to learn from data.\nIt identifies patterns and supports predictions.\n'
Section: ['Machine Learning']
Offsets: (20, 122)

Block 3
Type: heading
Content: '## Learning Algorithms'
Section: ['Machine Learning', 'Learning Algorithms']
Offsets: (123, 145)

Block 4
Type: unordered_list
Content: '- Supervised Learning\n- Unsupervised Learning'
Section: ['Machine Learning', 'Learning Algorithms']
Offsets: (147, 192)

Block 5
Type: heading
Content: '### Model Comparison'
Section: ['Machine Learning', 'Learning Algorithms', 'Model Comparison']
Offsets: (194, 214)

Block 6
Type: table
Content: '| Algorithm | Type |\n|---|---|\n| Linear Regression | Supervised |\n| K-Means | Unsupervised |'
Section: ['Machine Learning', 'Learning Algorithms', 'Model Comparison']
Offsets: (216, 308)

Block 7
Type: code
Conten

### Result Interpretation

The unified Markdown parser successfully integrated all seven structural detectors.

### Detection Results

A total of 12 structural blocks were identified:

| Block Type      |  Count |
| --------------- | -----: |
| Heading         |      3 |
| Paragraph       |      3 |
| Unordered List  |      1 |
| Ordered List    |      1 |
| Table           |      1 |
| Code            |      1 |
| Horizontal Rule |      1 |
| **Total**       | **12** |

### Section Hierarchy

The parser correctly maintained heading hierarchy and propagated section paths to subsequent blocks.

When a new heading appeared at an existing or higher level, the previous lower-level sections were removed from the active hierarchy.

### Source Integrity

All validation checks passed:

* Blocks are ordered by source position.
* No overlapping blocks exist.
* Character offsets match the original source.
* Code-fenced content is excluded from other structural classifications.
* All seven expected block types are present.

### Conclusion

The unified Markdown parser successfully combines individual structural detectors into a single deterministic parsing interface.

The implementation is ready for additional edge-case validation before integration with the recursive chunking strategy.


## Markdown Parser Edge-Case Validation

#### Objective

Validate the unified Markdown parser against complex and unusual Markdown structures.

#### Test Scenarios

1. Nested heading hierarchies and level transitions.
2. Consecutive Markdown tables.
3. Blank lines separating list items.
4. Unclosed and unusual fenced code blocks.
5. Arabic text and Unicode character offsets.
6. Unclassified source content and structural boundary preservation.

#### Validation Requirements

* Preserve exact source offsets.
* Maintain correct section paths.
* Prevent overlapping structural blocks.
* Exclude code-fenced content from other detectors.
* Preserve Unicode characters without corrupting offsets.
* Identify unsupported or unclassified content explicitly.

#### Expected Outcome

A more reliable Markdown parser with verified behavior across realistic document structures and edge cases.


In [ ]:
import app.chunking.markdown_parser as markdown_parser

headings = markdown_parser.detect_headings(markdown_sample)
blocks = markdown_parser.parse_markdown_blocks(markdown_sample)

print("HEADING HIERARCHY & UNICODE VALIDATION")
print("=" * 75)

for heading in headings:
    print(
        f"Level {heading.metadata['heading_level']} | "
        f"{heading.metadata['heading_text']}"
    )
    print(f"Section Path: {heading.section_path}")
    print(f"Offsets: ({heading.char_start}, {heading.char_end})")
    print("-" * 75)


expected_paths = [
    ["Machine Learning"],
    ["Machine Learning", "Deep Learning"],
    ["Machine Learning", "Deep Learning", "Transformers"],
    ["Machine Learning", "Computer Vision"],
    ["Machine Learning", "Computer Vision", "Object Detection"],
    ["هندسة الذكاء الاصطناعي"],
    ["هندسة الذكاء الاصطناعي", "معالجة اللغة الطبيعية"],
    ["هندسة الذكاء الاصطناعي", "Final Section"],
]

actual_paths = [heading.section_path for heading in headings]

print("\nVALIDATION RESULTS")
print("=" * 75)

print("Expected headings:", len(expected_paths))
print("Actual headings:", len(headings))

assert len(headings) == 8, f"Expected 8 headings, got {len(headings)}"

assert actual_paths == expected_paths, (
    f"Heading hierarchy mismatch.\nExpected: {expected_paths}\nActual: {actual_paths}"
)

assert not any(
    "Fake Heading" in heading.content or "Another Fake Heading" in heading.content
    for heading in headings
)

arabic_headings = [
    heading
    for heading in headings
    if any("\u0600" <= char <= "\u06ff" for char in heading.content)
]

assert len(arabic_headings) == 2

for heading in headings:
    assert markdown_sample[heading.char_start : heading.char_end] == heading.content

for block in blocks:
    assert markdown_sample[block.char_start : block.char_end] == block.content

for current, following in zip(blocks, blocks[1:]):
    assert current.char_end <= following.char_start

print("\nTOTAL HEADINGS:", len(headings))
print("TOTAL PARSED BLOCKS:", len(blocks))
print("ARABIC HEADINGS:", len(arabic_headings))
print("ALL HEADING AND UNICODE ASSERTIONS PASSED")

HEADING HIERARCHY & UNICODE VALIDATION
Level 1 | Machine Learning
Section Path: ['Machine Learning']
Offsets: (0, 18)
---------------------------------------------------------------------------
Level 2 | Deep Learning
Section Path: ['Machine Learning', 'Deep Learning']
Offsets: (54, 70)
---------------------------------------------------------------------------
Level 3 | Transformers
Section Path: ['Machine Learning', 'Deep Learning', 'Transformers']
Offsets: (108, 124)
---------------------------------------------------------------------------
Level 2 | Computer Vision
Section Path: ['Machine Learning', 'Computer Vision']
Offsets: (165, 183)
---------------------------------------------------------------------------
Level 4 | Object Detection
Section Path: ['Machine Learning', 'Computer Vision', 'Object Detection']
Offsets: (223, 244)
---------------------------------------------------------------------------
Level 1 | هندسة الذكاء الاصطناعي
Section Path: ['هندسة الذكاء الاصطناعي']
Of

### Result Interpretation

The heading hierarchy and Unicode validation completed successfully.

**Results:**

* Total headings detected: 8
* Total parsed blocks: 17
* Arabic headings detected: 2
* Heading levels validated: H1, H2, H3, and H4
* All section paths matched the expected hierarchy.
* Headings inside fenced code blocks were correctly ignored.
* Character offsets were preserved correctly.

**Key Observations:**

1. The parser correctly maintains parent-child relationships between heading levels.
2. A new H1 heading resets the previous hierarchy.
3. H2 headings remain nested under the most recent H1.
4. Arabic headings are handled correctly without Unicode corruption.
5. Skipped heading levels, such as H2 followed by H4, are represented consistently in the section path.

**Conclusion:**

The heading detection component passed all hierarchy and Unicode assertions. No changes to the parser implementation were required.


### Consecutive Markdown Tables

### Objective

Validate that the Markdown parser correctly detects multiple tables within the same document without merging them into a single table.

### Test Scenarios

1. Detect two separate Markdown tables in one document.
2. Preserve each table's original content and character offsets.
3. Extract the correct table boundaries.
4. Preserve the correct section path for each table.
5. Ensure that the unified parser classifies both tables as `table` blocks.
6. Verify that no overlapping table ranges are produced.

### Expected Outcome

The parser should detect exactly two independent tables:

* Table 1 under `Dataset Overview`.
* Table 2 under `Dataset Overview > Model Metrics`.

Both tables should retain their original content, correct offsets, and section hierarchy.


In [ ]:
from app.chunking.markdown_parser import (
    detect_markdown_tables,
    parse_markdown_blocks,
)

markdown_sample = """# Dataset Overview

The following table describes the training dataset.

| Feature | Type | Description |
|---|---|---|
| age | integer | User age |
| income | float | Annual income |

## Model Metrics

The second table contains evaluation results.

| Metric | Value |
|---|---|
| Accuracy | 0.95 |
| F1 Score | 0.92 |
"""

tables = detect_markdown_tables(markdown_sample)
blocks = parse_markdown_blocks(markdown_sample)

print("=" * 75)
print("CONSECUTIVE MARKDOWN TABLES VALIDATION")
print("=" * 75)

for index, table in enumerate(tables, start=1):
    print(f"\nTABLE {index}")
    print("-" * 75)
    print("Section Path:", table.section_path)
    print("Offsets:", (table.char_start, table.char_end))
    print("Content:\n", table.content)

assert len(tables) == 2, f"Expected 2 tables, got {len(tables)}"

assert tables[0].section_path == ["Dataset Overview"]
assert tables[1].section_path == ["Dataset Overview", "Model Metrics"]

assert all(table.block_type == "table" for table in tables)

assert "Feature" in tables[0].content
assert "Accuracy" in tables[1].content

assert tables[0].char_end <= tables[1].char_start

for table in tables:
    assert markdown_sample[table.char_start : table.char_end] == table.content

parsed_tables = [block for block in blocks if block.block_type == "table"]

assert len(parsed_tables) == 2

print("\nVALIDATION RESULTS")
print("=" * 75)
print("Expected tables: 2")
print("Actual tables:", len(tables))
print("Parsed table blocks:", len(parsed_tables))
print("All table assertions passed.")

CONSECUTIVE MARKDOWN TABLES VALIDATION

TABLE 1
---------------------------------------------------------------------------
Section Path: ['Dataset Overview']
Offsets: (73, 183)
Content:
 | Feature | Type | Description |
|---|---|---|
| age | integer | User age |
| income | float | Annual income |

TABLE 2
---------------------------------------------------------------------------
Section Path: ['Dataset Overview', 'Model Metrics']
Offsets: (250, 318)
Content:
 | Metric | Value |
|---|---|
| Accuracy | 0.95 |
| F1 Score | 0.92 |

VALIDATION RESULTS
Expected tables: 2
Actual tables: 2
Parsed table blocks: 2
All table assertions passed.


#### Result Interpretation

The consecutive Markdown tables validation completed successfully.

**Results:**

* Expected tables: 2
* Detected tables: 2
* Tables classified by the unified parser: 2
* Table 1 section path: `Dataset Overview`
* Table 2 section path: `Dataset Overview > Model Metrics`

**Key Observations:**

1. Both Markdown tables were detected independently.
2. The parser correctly preserved table boundaries.
3. Each table retained its original content.
4. Character offsets matched the corresponding source text.
5. Section paths correctly reflected the heading hierarchy.
6. No overlapping table ranges were detected.

**Conclusion:**

The Markdown table detection component passed all assertions. Multiple tables within the same document are correctly classified and associated with their respective sections.


### Code Fence Edge Cases

### Objective

Validate the parser's ability to detect fenced code blocks under different Markdown formatting conditions.

### Test Scenarios

1. Detect standard backtick code fences.
2. Detect tilde-based code fences.
3. Handle closing fences that contain more markers than the opening fence.
4. Handle unclosed code fences extending to the end of the document.
5. Ignore headings, tables, and horizontal rules inside fenced code blocks.
6. Preserve language metadata and source character offsets.

### Expected Outcome

The parser should correctly identify all valid and unclosed code fences while preventing their internal content from being classified as independent Markdown structures.


In [ ]:
from app.chunking.markdown_parser import (
    detect_code_fences,
    detect_code_blocks,
    parse_markdown_blocks,
)

fence = "`" * 3
long_fence = "`" * 4

markdown_sample = "\n".join(
    [
        "# Code Fence Tests",
        "",
        fence + "python",
        'print("Hello World")',
        "# This is not a heading",
        fence,
        "",
        "~~~json",
        "{",
        '    "name": "test",',
        '    "value": 42',
        "}",
        "~~~~",
        "",
        long_fence + "javascript",
        'const message = "Hello";',
        "// ## Fake Heading",
        long_fence + "`",
        "",
        "## After Code",
        "",
        fence + "python",
        "# Unclosed code fence",
        'print("Still inside code")',
    ]
)

fences = detect_code_fences(markdown_sample)
code_blocks = detect_code_blocks(markdown_sample)
all_blocks = parse_markdown_blocks(markdown_sample)

print("=" * 75)
print("CODE FENCE EDGE CASES VALIDATION")
print("=" * 75)

for index, item in enumerate(fences, start=1):
    print(f"\nFENCE {index}")
    print("-" * 75)
    print("Fence Character:", repr(item.fence_char))
    print("Fence Length:", item.fence_length)
    print("Language:", item.info_string)
    print("Closed:", item.closed)
    print("Opening Line:", item.opening_line)
    print("Closing Line:", item.closing_line)
    print("Offsets:", (item.char_start, item.char_end))

assert len(fences) == 4, f"Expected 4 fences, got {len(fences)}"

assert fences[0].fence_char == "`"
assert fences[0].fence_length == 3
assert fences[0].info_string == "python"
assert fences[0].closed is True

assert fences[1].fence_char == "~"
assert fences[1].fence_length == 3
assert fences[1].info_string == "json"
assert fences[1].closed is True

assert fences[2].fence_char == "`"
assert fences[2].fence_length == 4
assert fences[2].info_string == "javascript"
assert fences[2].closed is True

assert fences[3].fence_char == "`"
assert fences[3].fence_length == 3
assert fences[3].info_string == "python"
assert fences[3].closed is False

assert len(code_blocks) == 4
assert all(block.block_type == "code" for block in code_blocks)

fake_headings = [
    block
    for block in all_blocks
    if block.block_type == "heading" and "Fake Heading" in block.content
]

assert len(fake_headings) == 0

print("\nVALIDATION RESULTS")
print("=" * 75)
print("Expected fences: 4")
print("Detected fences:", len(fences))
print("Code blocks:", len(code_blocks))
print("Unclosed fences:", sum(not item.closed for item in fences))
print("Fake headings detected:", len(fake_headings))
print("All code fence assertions passed.")

CODE FENCE EDGE CASES VALIDATION

FENCE 1
---------------------------------------------------------------------------
Fence Character: '`'
Fence Length: 3
Language: python
Closed: True
Opening Line: 3
Closing Line: 6
Offsets: (20, 79)

FENCE 2
---------------------------------------------------------------------------
Fence Character: '~'
Fence Length: 3
Language: json
Closed: True
Opening Line: 8
Closing Line: 13
Offsets: (80, 133)

FENCE 3
---------------------------------------------------------------------------
Fence Character: '`'
Fence Length: 4
Language: javascript
Closed: True
Opening Line: 15
Closing Line: 18
Offsets: (134, 199)

FENCE 4
---------------------------------------------------------------------------
Fence Character: '`'
Fence Length: 3
Language: python
Closed: False
Opening Line: 22
Closing Line: None
Offsets: (215, 273)

VALIDATION RESULTS
Expected fences: 4
Detected fences: 4
Code blocks: 4
Unclosed fences: 1
Fake headings detected: 0
All code fence assertions 

#### Result Interpretation

The code fence edge-case validation completed successfully.

**Results:**

* Total detected fences: 4
* Total classified code blocks: 4
* Closed fences: 3
* Unclosed fences: 1
* Fake headings detected: 0

**Key Observations:**

1. Backtick-based and tilde-based fences were detected correctly.
2. Different opening fence lengths were preserved.
3. Closing fences with additional markers were handled correctly.
4. Unclosed fences extended to the end of the document.
5. Markdown headings inside fenced code blocks were not detected as independent headings.

**Conclusion:**

The code fence detection component passed all assertions, including marker type, marker length, language information, closing status, and source boundaries.


### Code Fence Closing Rules

### Objective

Verify that the parser applies the correct closing rules for fenced code blocks.

### Test Scenarios

1. A closing fence with the same marker type and length as the opening fence.
2. A closing fence with the same marker type but a greater length.
3. A closing fence using a different marker type.
4. A closing fence shorter than the opening fence.

### Expected Outcome

A fence should only close when the closing marker uses the same character as the opening marker and contains at least the same number of markers.

Invalid closing markers should not terminate the code block.


In [ ]:
from app.chunking.markdown_parser import detect_code_fences

fence_cases = {
    "same_length": (
        "```python\nprint('test')\n```",
        True,
    ),
    "longer_closing": (
        "````python\nprint('test')\n`````",
        True,
    ),
    "different_marker": (
        "```python\nprint('test')\n~~~",
        False,
    ),
    "shorter_closing": (
        "````python\nprint('test')\n```",
        False,
    ),
}

print("=" * 75)
print("CODE FENCE CLOSING RULES VALIDATION")
print("=" * 75)

for case_name, (sample, expected_closed) in fence_cases.items():
    detected = detect_code_fences(sample)

    assert len(detected) == 1, f"{case_name}: Expected 1 fence, got {len(detected)}"

    actual_closed = detected[0].closed

    print(f"\nTest: {case_name}")
    print("-" * 75)
    print("Fence Character:", repr(detected[0].fence_char))
    print("Opening Length:", detected[0].fence_length)
    print("Closed:", actual_closed)
    print("Expected:", expected_closed)

    assert actual_closed == expected_closed, (
        f"{case_name}: Expected closed={expected_closed}, got closed={actual_closed}"
    )

print("\nVALIDATION RESULTS")
print("=" * 75)
print("Total test cases:", len(fence_cases))
print("Passed:", len(fence_cases))
print("Failed: 0")
print("All code fence closing rule assertions passed.")

CODE FENCE CLOSING RULES VALIDATION

Test: same_length
---------------------------------------------------------------------------
Fence Character: '`'
Opening Length: 3
Closed: True
Expected: True

Test: longer_closing
---------------------------------------------------------------------------
Fence Character: '`'
Opening Length: 4
Closed: True
Expected: True

Test: different_marker
---------------------------------------------------------------------------
Fence Character: '`'
Opening Length: 3
Closed: False
Expected: False

Test: shorter_closing
---------------------------------------------------------------------------
Fence Character: '`'
Opening Length: 4
Closed: False
Expected: False

VALIDATION RESULTS
Total test cases: 4
Passed: 4
Failed: 0
All code fence closing rule assertions passed.


#### Result Interpretation

The code fence closing rules validation completed successfully.

**Results:**

* Total test cases: 4
* Passed: 4
* Failed: 0

**Key Observations:**

1. A closing fence with the same marker type and length is accepted.
2. A longer closing fence using the same marker type is accepted.
3. A closing fence using a different marker type is rejected.
4. A closing fence shorter than the opening fence is rejected.

**Conclusion:**

The parser correctly enforces Markdown code fence closing rules. A code block is only considered closed when the closing marker uses the same character and has a length equal to or greater than the opening marker.


### Fence Indentation & Language Metadata

### Objective

Validate how the parser handles indentation around fenced code blocks and extracts language information from opening fences.

### Test Scenarios

1. Detect a code fence with no indentation.
2. Detect a code fence with one to three leading spaces.
3. Extract language identifiers such as `python`, `javascript`, and `json`.
4. Preserve the correct opening and closing boundaries.
5. Verify that code content remains classified as a code block.

### Expected Outcome

The parser should correctly detect supported fenced code blocks, preserve their source boundaries, and extract the language information available in the opening fence.


In [ ]:
from app.chunking.markdown_parser import (
    detect_code_fences,
    detect_code_blocks,
)

fence = "`" * 3

markdown_sample = "\n".join(
    [
        "# Fence Indentation Tests",
        "",
        fence + "python",
        "print('No indentation')",
        fence,
        "",
        " " + fence + "javascript",
        "console.log('One space');",
        " " + fence,
        "",
        "  " + fence + "json",
        '{"name": "test"}',
        "  " + fence,
        "",
        "   " + fence + "python",
        "print('Three spaces')",
        "   " + fence,
    ]
)

fences = detect_code_fences(markdown_sample)
code_blocks = detect_code_blocks(markdown_sample)

print("=" * 75)
print("FENCE INDENTATION & LANGUAGE METADATA VALIDATION")
print("=" * 75)

for index, item in enumerate(fences, start=1):
    print(f"\nFENCE {index}")
    print("-" * 75)
    print("Language:", item.info_string)
    print("Fence Character:", repr(item.fence_char))
    print("Fence Length:", item.fence_length)
    print("Opening Line:", item.opening_line)
    print("Closing Line:", item.closing_line)
    print("Closed:", item.closed)
    print("Offsets:", (item.char_start, item.char_end))

expected_languages = [
    "python",
    "javascript",
    "json",
    "python",
]

assert len(fences) == 4, f"Expected 4 fences, got {len(fences)}"

assert [item.info_string for item in fences] == expected_languages

assert all(item.closed for item in fences)

assert len(code_blocks) == 4

assert all(block.block_type == "code" for block in code_blocks)

for item in fences:
    assert item.char_start < item.char_end
    assert markdown_sample[item.char_start : item.char_end]

print("\nVALIDATION RESULTS")
print("=" * 75)
print("Expected fences: 4")
print("Detected fences:", len(fences))
print("Expected languages:", expected_languages)
print("Actual languages:", [item.info_string for item in fences])
print("Code blocks:", len(code_blocks))
print("Closed fences:", sum(item.closed for item in fences))
print("All indentation and metadata assertions passed.")

FENCE INDENTATION & LANGUAGE METADATA VALIDATION

FENCE 1
---------------------------------------------------------------------------
Language: python
Fence Character: '`'
Fence Length: 3
Opening Line: 3
Closing Line: 5
Closed: True
Offsets: (27, 65)

FENCE 2
---------------------------------------------------------------------------
Language: javascript
Fence Character: '`'
Fence Length: 3
Opening Line: 7
Closing Line: 9
Closed: True
Offsets: (66, 112)

FENCE 3
---------------------------------------------------------------------------
Language: json
Fence Character: '`'
Fence Length: 3
Opening Line: 11
Closing Line: 13
Closed: True
Offsets: (113, 146)

FENCE 4
---------------------------------------------------------------------------
Language: python
Fence Character: '`'
Fence Length: 3
Opening Line: 15
Closing Line: 17
Closed: True
Offsets: (147, 188)

VALIDATION RESULTS
Expected fences: 4
Detected fences: 4
Expected languages: ['python', 'javascript', 'json', 'python']
Actual lang

#### Result Interpretation

The fence indentation and language metadata validation completed successfully.

**Results:**

* Expected code fences: 4
* Detected code fences: 4
* Classified code blocks: 4
* Successfully closed fences: 4
* Language metadata extraction: Passed
* Source character offsets: Valid

**Language Metadata:**

| Fence | Expected Language | Detected Language |
| ----- | ----------------- | ----------------- |
| 1     | Python            | Python            |
| 2     | JavaScript        | JavaScript        |
| 3     | JSON              | JSON              |
| 4     | Python            | Python            |

**Key Observations:**

1. The parser correctly detected code fences with zero to three leading spaces.
2. Language identifiers were extracted correctly.
3. All four code fences were recognized as closed.
4. Character offsets were valid and preserved.
5. All detected code blocks were classified as `code`.

**Conclusion:**

The fence indentation and language metadata validation passed all assertions. No changes to the parser implementation were required.


## Final Regression Test

The complete project test suite was executed successfully using `pytest -v`.

### Test Summary

* Total tests: 480
* Passed: 480
* Failed: 0
* Errors: 0
* Execution time: 43.79 seconds

### Coverage Areas

The regression suite covered:

* Markdown and chunking models.
* Document discovery and ingestion.
* PDF and text extraction.
* Document processing and rollback.
* Runtime configuration and validation.
* Conversation management and prompt building.
* Ollama runtime and streaming.
* Structured output and token counting.
* CLI functionality.
* Web API, sessions, security, and request handling.

### Conclusion

All 480 tests passed successfully.

The Markdown Parser implementation completed its final regression validation without introducing failures in the existing project test suite.

# 9. Recursive Structure-Aware Chunking

This section implements a recursive chunking strategy that preserves the logical structure and contextual relationships of source documents. It uses the existing Markdown parser to identify structural boundaries and applies a hierarchical splitting approach to produce coherent, traceable chunks while respecting the configured character-size limits and preserving source offsets and section metadata.

## Implementation Design

The recursive structure-aware chunking strategy will be implemented by reusing the existing Markdown parser and chunking components.

The implementation will follow a hierarchical splitting approach that prioritizes document structure over arbitrary character boundaries. It will preserve section context, source offsets, and document provenance while ensuring that generated chunks respect the configured size and overlap constraints.

Each page will be processed independently to prevent chunks from crossing document or page boundaries. The resulting chunks will retain the metadata required for traceability, validation, and comparison with the fixed-window baseline.
